In [193]:
# ==========================================
# NOTEBOOK 07 — RECOMMENDATION ENGINE
# ==========================================

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

users = pd.read_csv(
    "../data/processed/users_clean.csv"
)

creators = pd.read_csv(
    "../data/processed/creators_clean.csv"
)

content = pd.read_csv(
    "../data/processed/content_clean.csv"
)

interactions = pd.read_csv(
    "../data/processed/interactions_clean.csv"
)

users["signup_date"] = pd.to_datetime(
    users["signup_date"]
)

creators["signup_date"] = pd.to_datetime(
    creators["signup_date"]
)

content["created_at"] = pd.to_datetime(
    content["created_at"]
)

interactions["timestamp"] = pd.to_datetime(
    interactions["timestamp"]
)

print("Users:", users.shape)
print("Creators:", creators.shape)
print("Content:", content.shape)
print("Interactions:", interactions.shape)

Users: (5000, 7)
Creators: (500, 5)
Content: (10000, 9)
Interactions: (250000, 22)


In [194]:
# ==========================================
# VERIFY REQUIRED COLUMNS
# ==========================================

required_columns = {
    "users": [
        "user_id",
        "country",
        "age_group",
        "signup_date",
        "following_count",
        "creator_flag",
        "preferred_genres"
    ],

    "creators": [
        "creator_id",
        "creator_name",
        "signup_date",
        "creator_type",
        "followers"
    ],

    "content": [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration",
        "created_at"
    ],

    "interactions": [
        "user_id",
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration",
        "creator_followers",
        "content_created_at",
        "genre_match",
        "timestamp",
        "content_age_days",
        "freshness",
        "impression",
        "clicked",
        "watch_time",
        "completion_rate",
        "liked",
        "saved",
        "shared",
        "commented",
        "recreated",
        "meaningful_engagement"
    ]
}

for table_name, columns in required_columns.items():

    table = {
        "users": users,
        "creators": creators,
        "content": content,
        "interactions": interactions
    }[table_name]

    missing = [
        column
        for column in columns
        if column not in table.columns
    ]

    assert not missing, (
        f"{table_name} is missing columns: {missing}"
    )

print("✓ All required columns are present")

✓ All required columns are present


In [195]:
# ==========================================
# RECOMMENDATION CUTOFF
# ==========================================

RECOMMENDATION_CUTOFF = interactions[
    "timestamp"
].quantile(0.85)

historical_interactions = interactions[
    interactions["timestamp"] <= RECOMMENDATION_CUTOFF
].copy()

future_interactions = interactions[
    interactions["timestamp"] > RECOMMENDATION_CUTOFF
].copy()

print("Recommendation cutoff:")
print(RECOMMENDATION_CUTOFF)

print(
    "\nHistorical interactions:",
    len(historical_interactions)
)

print(
    "Future interactions:",
    len(future_interactions)
)

print(
    "\nHistorical period:",
    historical_interactions["timestamp"].min(),
    "→",
    historical_interactions["timestamp"].max()
)

print(
    "Future period:",
    future_interactions["timestamp"].min(),
    "→",
    future_interactions["timestamp"].max()
)

Recommendation cutoff:
2026-08-15 05:05:33.349999872

Historical interactions: 212500
Future interactions: 37500

Historical period: 2025-06-01 23:55:16 → 2026-08-15 05:05:32
Future period: 2026-08-15 05:05:41 → 2026-08-30 23:59:46


In [196]:
# ==========================================
# TEMPORAL INTEGRITY
# ==========================================

assert (
    historical_interactions["timestamp"].max()
    <= RECOMMENDATION_CUTOFF
)

assert (
    future_interactions["timestamp"].min()
    > RECOMMENDATION_CUTOFF
)

print("✓ Temporal separation verified")

✓ Temporal separation verified


In [197]:
# ==========================================
# EVALUATION USERS
# ==========================================

historical_users = set(
    historical_interactions["user_id"]
)

future_users = set(
    future_interactions["user_id"]
)

evaluation_users = sorted(
    historical_users.intersection(
        future_users
    )
)

future_eval_interactions = future_interactions[
    future_interactions["user_id"].isin(
        evaluation_users
    )
].copy()

print(
    "Users with historical activity:",
    len(historical_users)
)

print(
    "Users with future activity:",
    len(future_users)
)

print(
    "Evaluation users:",
    len(evaluation_users)
)

print(
    "Future interactions from evaluation users:",
    len(future_eval_interactions)
)

Users with historical activity: 4998
Users with future activity: 4780
Evaluation users: 4779
Future interactions from evaluation users: 37499


In [198]:
# ==========================================
# ELIGIBLE CONTENT
# ==========================================

eligible_content = content[
    content["created_at"] <= RECOMMENDATION_CUTOFF
].copy()

print(
    "Total content:",
    len(content)
)

print(
    "Eligible content:",
    len(eligible_content)
)

assert (
    eligible_content["created_at"].max()
    <= RECOMMENDATION_CUTOFF
)

print("✓ All candidate content existed at cutoff")

Total content: 10000
Eligible content: 9659
✓ All candidate content existed at cutoff


In [199]:
# ==========================================
# USER → SEEN CONTENT
# ==========================================

user_seen_content = (
    historical_interactions
    .groupby("user_id")["content_id"]
    .apply(set)
    .to_dict()
)

print(
    "Users with seen-content history:",
    len(user_seen_content)
)

test_user = evaluation_users[0]

print(
    "\nTest user:",
    test_user
)

print(
    "Previously seen content:",
    len(
        user_seen_content.get(
            test_user,
            set()
        )
    )
)

Users with seen-content history: 4998

Test user: U00001
Previously seen content: 51


In [200]:
# ==========================================
# GLOBAL POPULARITY STATISTICS
# ==========================================

global_content_stats = (
    historical_interactions
    .groupby("content_id")
    .agg(
        impressions=("impression", "sum"),
        clicks=("clicked", "sum"),
        meaningful_engagements=(
            "meaningful_engagement",
            "sum"
        ),
        watch_time=("watch_time", "sum"),
        avg_completion=(
            "completion_rate",
            "mean"
        )
    )
    .reset_index()
)

global_content_stats["engagement_rate"] = (
    global_content_stats[
        "meaningful_engagements"
    ]
    /
    global_content_stats[
        "impressions"
    ].replace(0, np.nan)
)

global_content_stats = global_content_stats.merge(
    eligible_content[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "created_at"
        ]
    ],
    on="content_id",
    how="right"
)

global_content_stats = global_content_stats.fillna({
    "impressions": 0,
    "clicks": 0,
    "meaningful_engagements": 0,
    "watch_time": 0,
    "avg_completion": 0,
    "engagement_rate": 0
})

global_content_stats = (
    global_content_stats
    .sort_values(
        "meaningful_engagements",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    global_content_stats.head(10)
)

  content_id  impressions  clicks  meaningful_engagements  watch_time  \
0   CT001010      59.0000 29.0000                 29.0000    613.3600   
1   CT003051      53.0000 28.0000                 28.0000  1,120.2300   
2   CT007858      43.0000 26.0000                 26.0000    232.7400   
3   CT009767      45.0000 25.0000                 25.0000    773.1000   
4   CT004242      54.0000 26.0000                 25.0000    214.7500   
5   CT006791      51.0000 25.0000                 25.0000    854.8700   
6   CT005668      50.0000 24.0000                 24.0000  1,064.1900   
7   CT002446      48.0000 24.0000                 24.0000    248.2100   
8   CT008086      42.0000 24.0000                 24.0000    138.8700   
9   CT001063      47.0000 24.0000                 24.0000    564.3800   

   avg_completion  engagement_rate creator_id        genre content_type  \
0          0.3300           0.4915      C0451      Romance        image   
1          0.3878           0.5283      C0068 

In [201]:
# ==========================================
# GLOBAL POPULARITY RECOMMENDER
# ==========================================

def recommend_popular(
    user_id,
    k=10,
    exclude_seen=True
):

    recommendations = (
        global_content_stats.copy()
    )

    if exclude_seen:

        seen = user_seen_content.get(
            user_id,
            set()
        )

        recommendations = recommendations[
            ~recommendations["content_id"].isin(
                seen
            )
        ]

    return recommendations.head(k).copy()

In [202]:
# ==========================================
# TEST GLOBAL POPULARITY
# ==========================================

popular_recommendations = recommend_popular(
    test_user,
    k=10
)

print(
    popular_recommendations[
        [
            "content_id",
            "genre",
            "content_type",
            "meaningful_engagements",
            "engagement_rate"
        ]
    ]
)

  content_id        genre content_type  meaningful_engagements  \
0   CT001010      Romance        image                 29.0000   
1   CT003051       Comedy        video                 28.0000   
2   CT007858    Animation         mini                 26.0000   
3   CT009767      Mystery        image                 25.0000   
4   CT004242       Action         mini                 25.0000   
5   CT006791       Sci-Fi         cine                 25.0000   
6   CT005668  Documentary         cine                 24.0000   
7   CT002446       Action         cine                 24.0000   
8   CT008086       Comedy         mini                 24.0000   
9   CT001063      Romance        image                 24.0000   

   engagement_rate  
0           0.4915  
1           0.5283  
2           0.6047  
3           0.5556  
4           0.4630  
5           0.4902  
6           0.4800  
7           0.5000  
8           0.5714  
9           0.5106  


In [203]:
# ==========================================
# RECENT POPULARITY
# ==========================================

RECENT_WINDOW_DAYS = 30

recent_start = (
    RECOMMENDATION_CUTOFF
    - pd.Timedelta(
        days=RECENT_WINDOW_DAYS
    )
)

recent_interactions = historical_interactions[
    historical_interactions["timestamp"]
    >= recent_start
].copy()

print(
    "Recent window:"
)

print(
    recent_start,
    "→",
    RECOMMENDATION_CUTOFF
)

print(
    "Recent interactions:",
    len(recent_interactions)
)

Recent window:
2026-07-16 05:05:33.349999872 → 2026-08-15 05:05:33.349999872
Recent interactions: 45397


In [204]:
# ==========================================
# RECENT CONTENT STATISTICS
# ==========================================

recent_content_stats = (
    recent_interactions
    .groupby("content_id")
    .agg(
        impressions=("impression", "sum"),
        clicks=("clicked", "sum"),
        meaningful_engagements=(
            "meaningful_engagement",
            "sum"
        ),
        watch_time=("watch_time", "sum"),
        avg_completion=(
            "completion_rate",
            "mean"
        )
    )
    .reset_index()
)

recent_content_stats["engagement_rate"] = (
    recent_content_stats[
        "meaningful_engagements"
    ]
    /
    recent_content_stats[
        "impressions"
    ].replace(0, np.nan)
)

recent_content_stats = recent_content_stats.merge(
    eligible_content[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "created_at"
        ]
    ],
    on="content_id",
    how="right"
)

recent_content_stats = recent_content_stats.fillna({
    "impressions": 0,
    "clicks": 0,
    "meaningful_engagements": 0,
    "watch_time": 0,
    "avg_completion": 0,
    "engagement_rate": 0
})

recent_content_stats = (
    recent_content_stats
    .sort_values(
        "meaningful_engagements",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    recent_content_stats.head(10)
)

  content_id  impressions  clicks  meaningful_engagements  watch_time  \
0   CT009210      30.0000 21.0000                 21.0000     50.0000   
1   CT002215      27.0000 18.0000                 18.0000    282.0800   
2   CT001900      29.0000 17.0000                 17.0000    442.3300   
3   CT002295      32.0000 16.0000                 16.0000    126.1600   
4   CT002868      29.0000 16.0000                 16.0000    502.4600   
5   CT008155      27.0000 15.0000                 15.0000    495.0100   
6   CT007293      31.0000 15.0000                 15.0000    128.0900   
7   CT005537      26.0000 17.0000                 15.0000    228.9900   
8   CT001930      26.0000 14.0000                 14.0000    621.7600   
9   CT006963      31.0000 14.0000                 14.0000     88.5900   

   avg_completion  engagement_rate creator_id        genre content_type  \
0          0.4902           0.7000      C0060       Action         mini   
1          0.4465           0.6667      C0285 

In [205]:
# ==========================================
# RECENT POPULARITY RECOMMENDER
# ==========================================

def recommend_recent_popular(
    user_id,
    k=10,
    exclude_seen=True
):

    recommendations = (
        recent_content_stats.copy()
    )

    if exclude_seen:

        seen = user_seen_content.get(
            user_id,
            set()
        )

        recommendations = recommendations[
            ~recommendations["content_id"].isin(
                seen
            )
        ]

    return recommendations.head(k).copy()

In [206]:
# ==========================================
# TEST RECENT POPULARITY
# ==========================================

recent_recommendations = (
    recommend_recent_popular(
        test_user,
        k=10
    )
)

print(
    recent_recommendations[
        [
            "content_id",
            "genre",
            "content_type",
            "meaningful_engagements",
            "engagement_rate"
        ]
    ]
)

   content_id        genre content_type  meaningful_engagements  \
0    CT009210       Action         mini                 21.0000   
1    CT002215      Mystery         cine                 18.0000   
3    CT002295       Comedy         mini                 16.0000   
4    CT002868      Mystery        image                 16.0000   
5    CT008155       Sci-Fi        video                 15.0000   
6    CT007293      Fantasy         mini                 15.0000   
7    CT005537      Fantasy         mini                 15.0000   
8    CT001930    Animation        video                 14.0000   
9    CT006963  Documentary         mini                 14.0000   
10   CT007014  Documentary         mini                 14.0000   

    engagement_rate  
0            0.7000  
1            0.6667  
3            0.5000  
4            0.5517  
5            0.5556  
6            0.4839  
7            0.5769  
8            0.5385  
9            0.4516  
10           0.5000  


In [207]:
# ==========================================
# GLOBAL VS RECENT POPULARITY
# ==========================================

global_top_ids = set(
    recommend_popular(
        test_user,
        k=10
    )["content_id"]
)

recent_top_ids = set(
    recommend_recent_popular(
        test_user,
        k=10
    )["content_id"]
)

popularity_overlap = len(
    global_top_ids.intersection(
        recent_top_ids
    )
)

print(
    "Global top-10:",
    len(global_top_ids)
)

print(
    "Recent top-10:",
    len(recent_top_ids)
)

print(
    "Top-10 overlap:",
    popularity_overlap
)

Global top-10: 10
Recent top-10: 10
Top-10 overlap: 0


In [208]:
# ==========================================
# POSITIVE ENGAGEMENT HISTORY
# ==========================================

positive_history = historical_interactions[
    historical_interactions[
        "meaningful_engagement"
    ] == 1
].copy()

print(
    "Meaningful engagements:",
    len(positive_history)
)

print(
    "Users with meaningful engagement:",
    positive_history[
        "user_id"
    ].nunique()
)

Meaningful engagements: 75118
Users with meaningful engagement: 4949


In [209]:
# ==========================================
# CONTENT REPRESENTATION
# ==========================================

content_features = eligible_content[
    [
        "content_id",
        "genre",
        "content_type",
        "duration"
    ]
].copy()

# Reset index so DataFrame row positions
# exactly match TF-IDF matrix row positions
content_features = content_features.reset_index(drop=True)

# Create duration buckets
content_features["duration_bucket"] = pd.cut(
    content_features["duration"],
    bins=[0, 10, 20, 30, 60, 120, 300, np.inf],
    labels=[
        "very_short",
        "short",
        "medium",
        "long",
        "very_long",
        "extended",
        "very_extended"
    ],
    include_lowest=True
)

# Convert categorical values into explicit tokens
content_features["genre_token"] = (
    content_features["genre"]
    .str.lower()
    .str.replace("-", "_", regex=False)
)

content_features["type_token"] = (
    content_features["content_type"]
    .str.lower()
)

content_features["duration_token"] = (
    content_features["duration_bucket"]
    .astype(str)
    .str.lower()
)

# Build interpretable text representation
content_features["content_text"] = (
    "genre_" + content_features["genre_token"]
    + " type_" + content_features["type_token"]
    + " duration_" + content_features["duration_token"]
)

# TF-IDF representation
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"
)

content_matrix = vectorizer.fit_transform(
    content_features["content_text"]
)

# IMPORTANT:
# Matrix row position == content_features row position
content_id_to_index = pd.Series(
    content_features.index,
    index=content_features["content_id"]
)

print("Content feature rows:", len(content_features))
print("TF-IDF matrix shape:", content_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

print("\nFirst vocabulary terms:")
print(
    sorted(vectorizer.vocabulary_.keys())[:30]
)

# Integrity checks
assert content_matrix.shape[0] == len(content_features)

assert (
    content_id_to_index.min() == 0
)

assert (
    content_id_to_index.max()
    == len(content_features) - 1
)

print("\n✓ Content representation created")
print("✓ TF-IDF row positions aligned with content indices")

Content feature rows: 9659
TF-IDF matrix shape: (9659, 21)
Vocabulary size: 21

First vocabulary terms:
['duration_extended', 'duration_long', 'duration_medium', 'duration_short', 'duration_very_extended', 'duration_very_long', 'duration_very_short', 'genre_action', 'genre_animation', 'genre_comedy', 'genre_documentary', 'genre_drama', 'genre_fantasy', 'genre_horror', 'genre_mystery', 'genre_romance', 'genre_sci_fi', 'type_cine', 'type_image', 'type_mini', 'type_video']

✓ Content representation created
✓ TF-IDF row positions aligned with content indices


In [210]:
# ==========================================
# BUILD USER PROFILE
# ==========================================

def build_user_profile(user_id):

    user_history = positive_history[
        positive_history["user_id"] == user_id
    ]

    if user_history.empty:
        return None

    valid_content_ids = user_history[
        user_history["content_id"].isin(
            content_id_to_index.index
        )
    ]["content_id"]

    if valid_content_ids.empty:
        return None

    content_indices = (
        content_id_to_index
        .loc[valid_content_ids]
        .values
    )

    user_vectors = content_matrix[
        content_indices
    ]

    profile_vector = np.asarray(
        user_vectors.mean(axis=0)
    ).reshape(1, -1)

    return profile_vector

In [211]:
# ==========================================
# TEST USER PROFILE
# ==========================================

user_profile = build_user_profile(
    test_user
)

print(
    "User profile type:",
    type(user_profile)
)

print(
    "User profile shape:",
    user_profile.shape
)

User profile type: <class 'numpy.ndarray'>
User profile shape: (1, 21)


In [212]:
# ==========================================
# CONTENT SIMILARITY
# ==========================================

from sklearn.metrics.pairwise import (
    cosine_similarity
)

content_similarity_scores = cosine_similarity(
    user_profile,
    content_matrix
).flatten()

print(
    "Number of similarity scores:",
    len(content_similarity_scores)
)

print(
    "Min similarity:",
    content_similarity_scores.min()
)

print(
    "Max similarity:",
    content_similarity_scores.max()
)

Number of similarity scores: 9659
Min similarity: 0.06187941426845132
Max similarity: 0.5464931448421919


In [213]:
# ==========================================
# RANK CONTENT BY SIMILARITY
# ==========================================

content_ranked = content_features.copy()

content_ranked[
    "similarity_score"
] = content_similarity_scores

content_ranked = (
    content_ranked
    .sort_values(
        "similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    content_ranked[
        [
            "content_id",
            "genre",
            "content_type",
            "duration",
            "duration_bucket",
            "similarity_score"
        ]
    ].head(15)
)

   content_id   genre content_type  duration duration_bucket  similarity_score
0    CT009881  Comedy         mini   21.2000          medium            0.5465
1    CT007422  Comedy         mini   29.4000          medium            0.5465
2    CT007309  Comedy         mini   22.1000          medium            0.5465
3    CT001481  Comedy         mini   28.1000          medium            0.5465
4    CT002984  Comedy         mini   28.3000          medium            0.5465
5    CT001430  Comedy         mini   22.9000          medium            0.5465
6    CT007478  Comedy         mini   20.1000          medium            0.5465
7    CT005118  Comedy         mini   21.3000          medium            0.5465
8    CT009735  Comedy         mini   20.3000          medium            0.5465
9    CT008379  Comedy         mini   29.7000          medium            0.5465
10   CT001425  Comedy         mini   25.3000          medium            0.5465
11   CT003825  Comedy         mini   21.9000        

In [214]:
# ==========================================
# REMOVE SEEN CONTENT
# ==========================================

seen_content = set(
    historical_interactions.loc[
        historical_interactions["user_id"]
        == test_user,
        "content_id"
    ]
)

content_ranked_unseen = (
    content_ranked[
        ~content_ranked["content_id"].isin(
            seen_content
        )
    ]
    .copy()
)

print(
    "Seen content:",
    len(seen_content)
)

print(
    "Candidate content after filtering:",
    len(content_ranked_unseen)
)

print(
    "\nTop unseen personalized candidates:"
)

print(
    content_ranked_unseen[
        [
            "content_id",
            "genre",
            "content_type",
            "duration",
            "duration_bucket",
            "similarity_score"
        ]
    ].head(10)
)

Seen content: 51
Candidate content after filtering: 9608

Top unseen personalized candidates:
  content_id   genre content_type  duration duration_bucket  similarity_score
0   CT009881  Comedy         mini   21.2000          medium            0.5465
1   CT007422  Comedy         mini   29.4000          medium            0.5465
2   CT007309  Comedy         mini   22.1000          medium            0.5465
3   CT001481  Comedy         mini   28.1000          medium            0.5465
4   CT002984  Comedy         mini   28.3000          medium            0.5465
5   CT001430  Comedy         mini   22.9000          medium            0.5465
6   CT007478  Comedy         mini   20.1000          medium            0.5465
7   CT005118  Comedy         mini   21.3000          medium            0.5465
8   CT009735  Comedy         mini   20.3000          medium            0.5465
9   CT008379  Comedy         mini   29.7000          medium            0.5465


In [215]:
# ==========================================
# CONTENT-BASED RECOMMENDER DIAGNOSTIC
# ==========================================

print("Unique similarity scores in top 50:")
print(
    content_ranked_unseen["similarity_score"]
    .head(50)
    .value_counts()
)

print("\nTop similarity score:")
print(
    content_ranked_unseen["similarity_score"].max()
)

print("\nNumber of unique similarity scores:")
print(
    content_ranked_unseen["similarity_score"]
    .nunique()
)

print("\nTop 20 recommendations:")
print(
    content_ranked_unseen[
        [
            "content_id",
            "genre",
            "content_type",
            "duration",
            "duration_bucket",
            "similarity_score"
        ]
    ].head(20)
)

Unique similarity scores in top 50:
similarity_score
0.5465    50
Name: count, dtype: int64

Top similarity score:
0.5464931448421919

Number of unique similarity scores:
241

Top 20 recommendations:
   content_id   genre content_type  duration duration_bucket  similarity_score
0    CT009881  Comedy         mini   21.2000          medium            0.5465
1    CT007422  Comedy         mini   29.4000          medium            0.5465
2    CT007309  Comedy         mini   22.1000          medium            0.5465
3    CT001481  Comedy         mini   28.1000          medium            0.5465
4    CT002984  Comedy         mini   28.3000          medium            0.5465
5    CT001430  Comedy         mini   22.9000          medium            0.5465
6    CT007478  Comedy         mini   20.1000          medium            0.5465
7    CT005118  Comedy         mini   21.3000          medium            0.5465
8    CT009735  Comedy         mini   20.3000          medium            0.5465
9    CT008

In [216]:
# ==========================================
# CONTENT-BASED PERSONALIZATION CHECK
# ==========================================

# User's historically engaged genres
user_positive_genres = (
    positive_history[
        positive_history["user_id"] == test_user
    ]["genre"]
    .value_counts(normalize=True)
    .rename("history_share")
)

# Top 50 recommended genres
recommended_genres = (
    content_ranked_unseen
    .head(50)["genre"]
    .value_counts(normalize=True)
    .rename("recommendation_share")
)

genre_comparison = (
    pd.concat(
        [
            user_positive_genres,
            recommended_genres
        ],
        axis=1
    )
    .fillna(0)
    .sort_values(
        "recommendation_share",
        ascending=False
    )
)

print("Test user:", test_user)

print("\nGenre distribution comparison:")
print(
    genre_comparison
)

print("\nTop historical genres:")
print(
    user_positive_genres.head(5)
)

print("\nTop recommended genres:")
print(
    recommended_genres.head(5)
)

Test user: U00001

Genre distribution comparison:
             history_share  recommendation_share
genre                                           
Comedy              0.1500                1.0000
Documentary         0.1500                0.0000
Mystery             0.1500                0.0000
Action              0.1500                0.0000
Fantasy             0.1500                0.0000
Animation           0.1000                0.0000
Drama               0.1000                0.0000
Sci-Fi              0.0500                0.0000

Top historical genres:
genre
Documentary   0.1500
Action        0.1500
Mystery       0.1500
Fantasy       0.1500
Comedy        0.1500
Name: history_share, dtype: float64

Top recommended genres:
genre
Comedy   1.0000
Name: recommendation_share, dtype: float64


In [217]:
# ==========================================
# USER POSITIVE HISTORY PROFILE
# ==========================================

user_history = positive_history[
    positive_history["user_id"] == test_user
].copy()

print("Test user:", test_user)

print("\nNumber of positive interactions:")
print(len(user_history))

print("\nPositive interactions by genre:")
print(
    user_history["genre"]
    .value_counts()
)

print("\nPositive interactions by content type:")
print(
    user_history["content_type"]
    .value_counts()
)

print("\nPositive interactions by duration bucket:")

user_history["duration_bucket"] = pd.cut(
    user_history["duration"],
    bins=[0, 10, 20, 30, 60, 120, 300, np.inf],
    labels=[
        "very_short",
        "short",
        "medium",
        "long",
        "very_long",
        "extended",
        "very_extended"
    ],
    include_lowest=True
)

print(
    user_history["duration_bucket"]
    .value_counts()
)

Test user: U00001

Number of positive interactions:
20

Positive interactions by genre:
genre
Documentary    3
Action         3
Mystery        3
Fantasy        3
Comedy         3
Animation      2
Drama          2
Sci-Fi         1
Name: count, dtype: int64

Positive interactions by content type:
content_type
mini     10
image     4
video     3
cine      3
Name: count, dtype: int64

Positive interactions by duration bucket:
duration_bucket
medium           5
very_long        5
very_short       4
short            3
long             3
extended         0
very_extended    0
Name: count, dtype: int64


In [218]:
# ==========================================
# IMPROVED CONTENT REPRESENTATION
# ==========================================

content_features = eligible_content[
    [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration"
    ]
].copy()

# Reset index so matrix row positions
# align exactly with DataFrame positions
content_features = content_features.reset_index(drop=True)


# ------------------------------------------
# Duration bucket
# ------------------------------------------

content_features["duration_bucket"] = pd.cut(
    content_features["duration"],
    bins=[0, 10, 20, 30, 60, 120, 300, np.inf],
    labels=[
        "very_short",
        "short",
        "medium",
        "long",
        "very_long",
        "extended",
        "very_extended"
    ],
    include_lowest=True
)


# ------------------------------------------
# Explicit categorical tokens
# ------------------------------------------

content_features["genre_token"] = (
    content_features["genre"]
    .str.lower()
    .str.replace("-", "_", regex=False)
)

content_features["type_token"] = (
    content_features["content_type"]
    .str.lower()
)

content_features["duration_token"] = (
    content_features["duration_bucket"]
    .astype(str)
    .str.lower()
)

content_features["creator_token"] = (
    content_features["creator_id"]
    .str.lower()
)


# ------------------------------------------
# Combined content representation
# ------------------------------------------

content_features["content_text"] = (
    "genre_" + content_features["genre_token"]
    + " type_" + content_features["type_token"]
    + " duration_" + content_features["duration_token"]
    + " creator_" + content_features["creator_token"]
)


# ------------------------------------------
# TF-IDF
# ------------------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b"
)

content_matrix = vectorizer.fit_transform(
    content_features["content_text"]
)


# ------------------------------------------
# Content ID → matrix row mapping
# ------------------------------------------

content_id_to_index = pd.Series(
    content_features.index,
    index=content_features["content_id"]
)


# ------------------------------------------
# Validation
# ------------------------------------------

print(
    "Content feature rows:",
    len(content_features)
)

print(
    "TF-IDF matrix shape:",
    content_matrix.shape
)

print(
    "Vocabulary size:",
    len(vectorizer.vocabulary_)
)

print("\nSample vocabulary:")
print(
    sorted(vectorizer.vocabulary_.keys())[:30]
)


assert content_matrix.shape[0] == len(content_features)

assert content_id_to_index.min() == 0

assert (
    content_id_to_index.max()
    == len(content_features) - 1
)

print(
    "\n✓ Improved content representation created"
)

print(
    "✓ TF-IDF matrix aligned with content indices"
)

Content feature rows: 9659
TF-IDF matrix shape: (9659, 521)
Vocabulary size: 521

Sample vocabulary:
['creator_c0001', 'creator_c0002', 'creator_c0003', 'creator_c0004', 'creator_c0005', 'creator_c0006', 'creator_c0007', 'creator_c0008', 'creator_c0009', 'creator_c0010', 'creator_c0011', 'creator_c0012', 'creator_c0013', 'creator_c0014', 'creator_c0015', 'creator_c0016', 'creator_c0017', 'creator_c0018', 'creator_c0019', 'creator_c0020', 'creator_c0021', 'creator_c0022', 'creator_c0023', 'creator_c0024', 'creator_c0025', 'creator_c0026', 'creator_c0027', 'creator_c0028', 'creator_c0029', 'creator_c0030']

✓ Improved content representation created
✓ TF-IDF matrix aligned with content indices


In [219]:
# ==========================================
# REBUILD USER PROFILE
# ==========================================

user_profile = build_user_profile(
    test_user
)

print(
    "User profile type:",
    type(user_profile)
)

print(
    "User profile shape:",
    user_profile.shape
)

User profile type: <class 'numpy.ndarray'>
User profile shape: (1, 521)


In [220]:
# ==========================================
# RECALCULATE CONTENT SIMILARITY
# ==========================================

content_similarity_scores = cosine_similarity(
    user_profile,
    content_matrix
).flatten()

content_ranked = content_features.copy()

content_ranked["similarity_score"] = (
    content_similarity_scores
)

content_ranked = (
    content_ranked
    .sort_values(
        "similarity_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Number of similarity scores:",
    len(content_similarity_scores)
)

print(
    "Unique similarity scores:",
    content_ranked["similarity_score"].nunique()
)

print("\nTop content candidates:")

print(
    content_ranked[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "duration_bucket",
            "similarity_score"
        ]
    ].head(15)
)

Number of similarity scores: 9659
Unique similarity scores: 3779

Top content candidates:
   content_id creator_id        genre content_type  duration duration_bucket  \
0    CT005145      C0392  Documentary         mini   20.2000          medium   
1    CT004464      C0270       Comedy         mini   21.2000          medium   
2    CT001970      C0392      Mystery         mini   20.2000          medium   
3    CT001872      C0350       Comedy         mini   23.1000          medium   
4    CT009442      C0403      Mystery         mini   24.2000          medium   
5    CT003423      C0318      Mystery         mini   20.4000          medium   
6    CT002147      C0107       Action         mini   21.6000          medium   
7    CT008523      C0107       Action         mini   20.9000          medium   
8    CT003864      C0107  Documentary         mini   25.9000          medium   
9    CT009963      C0492      Fantasy         mini   20.4000          medium   
10   CT001165      C0127      

In [221]:
# ==========================================
# REMOVE SEEN CONTENT
# ==========================================

seen_content = set(
    historical_interactions.loc[
        historical_interactions["user_id"] == test_user,
        "content_id"
    ]
)

content_ranked_unseen = (
    content_ranked[
        ~content_ranked["content_id"].isin(
            seen_content
        )
    ]
    .copy()
)

print("Seen content:", len(seen_content))

print(
    "Candidate content after filtering:",
    len(content_ranked_unseen)
)

print("\nTop unseen personalized candidates:")

print(
    content_ranked_unseen[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "duration_bucket",
            "similarity_score"
        ]
    ].head(10)
)

Seen content: 51
Candidate content after filtering: 9608

Top unseen personalized candidates:
  content_id creator_id        genre content_type  duration duration_bucket  \
0   CT005145      C0392  Documentary         mini   20.2000          medium   
1   CT004464      C0270       Comedy         mini   21.2000          medium   
2   CT001970      C0392      Mystery         mini   20.2000          medium   
3   CT001872      C0350       Comedy         mini   23.1000          medium   
4   CT009442      C0403      Mystery         mini   24.2000          medium   
5   CT003423      C0318      Mystery         mini   20.4000          medium   
6   CT002147      C0107       Action         mini   21.6000          medium   
7   CT008523      C0107       Action         mini   20.9000          medium   
8   CT003864      C0107  Documentary         mini   25.9000          medium   
9   CT009963      C0492      Fantasy         mini   20.4000          medium   

   similarity_score  
0            0

In [222]:
# ==========================================
# CHECK SAVED MODEL ARTIFACTS
# ==========================================

from pathlib import Path

project_root = Path("..")

models_dir = project_root / "models"

print("Models directory:")
print(models_dir.resolve())

print("\nExists:", models_dir.exists())

if models_dir.exists():
    model_files = list(models_dir.rglob("*"))

    print("\nFiles found:")

    if model_files:
        for file in model_files:
            if file.is_file():
                print(
                    f"  {file.relative_to(models_dir)}"
                )
    else:
        print("  No files found.")

Models directory:
C:\Users\ADITYA\Documents\ds-projects\aicines-content-intelligence\models

Exists: True

Files found:
  engagement_feature_columns.joblib
  random_forest_engagement.joblib


In [223]:
# ==========================================
# LOAD SAVED ENGAGEMENT MODEL
# ==========================================

from pathlib import Path
import joblib

models_dir = Path("../models")

model_path = (
    models_dir /
    "random_forest_engagement.joblib"
)

feature_path = (
    models_dir /
    "engagement_feature_columns.joblib"
)

# Load complete preprocessing + model pipeline
random_forest = joblib.load(
    model_path
)

# Load frozen feature contract
feature_columns = joblib.load(
    feature_path
)

print("Model type:")
print(type(random_forest))

print("\nNumber of expected features:")
print(len(feature_columns))

print("\nModel path:")
print(model_path.resolve())

print("\n✓ Engagement model loaded")
print("✓ Feature contract loaded")

Model type:
<class 'sklearn.pipeline.Pipeline'>

Number of expected features:
53

Model path:
C:\Users\ADITYA\Documents\ds-projects\aicines-content-intelligence\models\random_forest_engagement.joblib

✓ Engagement model loaded
✓ Feature contract loaded


In [224]:
# ==========================================
# CREATE ML RANKING CANDIDATE POOL
# ==========================================

CANDIDATE_K = 200

ml_candidates = (
    content_ranked_unseen
    .head(CANDIDATE_K)
    .copy()
    .reset_index(drop=True)
)

print(
    "Candidate pool size:",
    len(ml_candidates)
)

print(
    "Expected:",
    CANDIDATE_K
)

print("\nCandidate columns:")
print(
    ml_candidates.columns.tolist()
)

print("\nFirst 10 candidates:")
print(
    ml_candidates[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "similarity_score"
        ]
    ].head(10)
)

assert len(ml_candidates) == CANDIDATE_K

assert (
    ml_candidates["content_id"]
    .isin(seen_content)
    .sum()
    == 0
)

print(
    "\n✓ Candidate pool created"
)

print(
    "✓ No previously seen content included"
)

Candidate pool size: 200
Expected: 200

Candidate columns:
['content_id', 'creator_id', 'genre', 'content_type', 'duration', 'duration_bucket', 'genre_token', 'type_token', 'duration_token', 'creator_token', 'content_text', 'similarity_score']

First 10 candidates:
  content_id creator_id        genre content_type  duration  similarity_score
0   CT005145      C0392  Documentary         mini   20.2000            0.3656
1   CT004464      C0270       Comedy         mini   21.2000            0.3650
2   CT001970      C0392      Mystery         mini   20.2000            0.3634
3   CT001872      C0350       Comedy         mini   23.1000            0.3631
4   CT009442      C0403      Mystery         mini   24.2000            0.3631
5   CT003423      C0318      Mystery         mini   20.4000            0.3626
6   CT002147      C0107       Action         mini   21.6000            0.3613
7   CT008523      C0107       Action         mini   20.9000            0.3613
8   CT003864      C0107  Documen

In [225]:
# ==========================================
# BUILD CANDIDATE MODEL FEATURES
# ==========================================

# Recreate the recommendation cutoff if it is not
# currently available in the notebook kernel.

cutoff = interactions["timestamp"].quantile(0.85)

# Historical interactions must be strictly before
# the recommendation cutoff.
historical_interactions = interactions[
    interactions["timestamp"] < cutoff
].copy()

history = historical_interactions.copy()

print("Historical interactions:", len(history))
print("ML candidates:", len(ml_candidates))
print("Recommendation cutoff:", cutoff)

# ------------------------------------------
# Candidate base table
# ------------------------------------------

candidate_features = ml_candidates[
    [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration"
    ]
].copy()

candidate_features["user_id"] = test_user

# ------------------------------------------
# Content metadata
# ------------------------------------------

candidate_features = candidate_features.merge(
    content[
        [
            "content_id",
            "created_at"
        ]
    ],
    on="content_id",
    how="left"
)

# ------------------------------------------
# Creator metadata
# ------------------------------------------

candidate_features = candidate_features.merge(
    creators[
        [
            "creator_id",
            "followers",
            "creator_type"
        ]
    ].rename(
        columns={
            "followers": "creator_followers"
        }
    ),
    on="creator_id",
    how="left"
)

# ------------------------------------------
# User metadata
# ------------------------------------------

candidate_features = candidate_features.merge(
    users[
        [
            "user_id",
            "country",
            "age_group",
            "signup_date",
            "following_count",
            "creator_flag"
        ]
    ],
    on="user_id",
    how="left"
)

print(
    "\nCandidate feature table shape:",
    candidate_features.shape
)

print("\nCandidate feature columns:")
print(
    candidate_features.columns.tolist()
)

print("\nFirst 5 rows:")
display(candidate_features.head())

print("\nMissing values:")
missing = candidate_features.isna().sum()
print(
    missing[missing > 0]
)

print("\n✓ Candidate base feature table created")

Historical interactions: 212500
ML candidates: 200
Recommendation cutoff: 2026-08-15 05:05:33.349999872

Candidate feature table shape: (200, 14)

Candidate feature columns:
['content_id', 'creator_id', 'genre', 'content_type', 'duration', 'user_id', 'created_at', 'creator_followers', 'creator_type', 'country', 'age_group', 'signup_date', 'following_count', 'creator_flag']

First 5 rows:


,content_id,creator_id,genre,content_type,duration,user_id,created_at,creator_followers,creator_type,country,age_group,signup_date,following_count,creator_flag
0,CT005145,C0392,Documentary,mini,20.2000,U00001,2026-02-19 05:06:03,321,Influencer,India,13-17,2024-02-08 19:21:10,15,True
1,CT004464,C0270,Comedy,mini,21.2000,U00001,2026-01-07 08:52:21,64,AI Creator,India,13-17,2024-02-08 19:21:10,15,True
2,CT001970,C0392,Mystery,mini,20.2000,U00001,2025-08-11 22:53:42,321,Influencer,India,13-17,2024-02-08 19:21:10,15,True
3,CT001872,C0350,Comedy,mini,23.1000,U00001,2026-02-03 05:25:07,10,Filmmaker,India,13-17,2024-02-08 19:21:10,15,True
4,CT009442,C0403,Mystery,mini,24.2000,U00001,2026-02-02 19:21:08,65,AI Creator,India,13-17,2024-02-08 19:21:10,15,True



Missing values:
Series([], dtype: int64)

✓ Candidate base feature table created


In [226]:
# ==========================================
# ADD USER HISTORICAL FEATURES
# ==========================================

# IMPORTANT:
# Only interactions before the recommendation cutoff
# are allowed to contribute to these features.

user_history = (
    history
    .groupby("user_id")
    .agg(
        user_prior_interactions=("user_id", "size"),
        user_prior_clicks=("clicked", "sum"),
        user_prior_engagements=("meaningful_engagement", "sum"),
        user_prior_watch_time=("watch_time", "sum"),
        user_prior_completions=("completion_rate", "sum"),
        user_prior_active_days=(
            "timestamp",
            lambda x: x.dt.date.nunique()
        ),
        user_prior_unique_content=(
            "content_id",
            "nunique"
        ),
        user_prior_unique_creators=(
            "creator_id",
            "nunique"
        )
    )
    .reset_index()
)

# ------------------------------------------
# Derived historical rates
# ------------------------------------------

user_history["user_prior_ctr"] = (
    user_history["user_prior_clicks"]
    / user_history["user_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

user_history["user_prior_engagement_rate"] = (
    user_history["user_prior_engagements"]
    / user_history["user_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

user_history["user_prior_avg_watch_time"] = (
    user_history["user_prior_watch_time"]
    / user_history["user_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

user_history["user_prior_avg_completion"] = (
    user_history["user_prior_completions"]
    / user_history["user_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

# ------------------------------------------
# Merge into candidate table
# ------------------------------------------

candidate_features = candidate_features.merge(
    user_history,
    on="user_id",
    how="left"
)

# ------------------------------------------
# Cold-start handling
# ------------------------------------------

user_behavior_columns = [
    "user_prior_interactions",
    "user_prior_clicks",
    "user_prior_engagements",
    "user_prior_watch_time",
    "user_prior_completions",
    "user_prior_active_days",
    "user_prior_unique_content",
    "user_prior_unique_creators",
    "user_prior_ctr",
    "user_prior_engagement_rate",
    "user_prior_avg_watch_time",
    "user_prior_avg_completion"
]

candidate_features[user_behavior_columns] = (
    candidate_features[user_behavior_columns]
    .fillna(0)
)

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print("\nUser historical features:")

display(
    candidate_features[
        ["user_id"] + user_behavior_columns
    ].drop_duplicates()
)

print("\n✓ User historical features added")

Candidate feature table shape: (200, 26)

User historical features:


,user_id,user_prior_interactions,user_prior_clicks,user_prior_engagements,user_prior_watch_time,user_prior_completions,user_prior_active_days,user_prior_unique_content,user_prior_unique_creators,user_prior_ctr,user_prior_engagement_rate,user_prior_avg_watch_time,user_prior_avg_completion
0,U00001,52,21,20,590.6500,15.3103,44,51,48,0.4038,0.3846,11.3587,0.2944



✓ User historical features added


In [227]:
# ==========================================
# ADD CREATOR HISTORICAL FEATURES
# ==========================================

creator_history = (
    history
    .groupby("creator_id")
    .agg(
        creator_prior_interactions=("creator_id", "size"),
        creator_prior_clicks=("clicked", "sum"),
        creator_prior_engagements=("meaningful_engagement", "sum")
    )
    .reset_index()
)

# ------------------------------------------
# Derived creator rates
# ------------------------------------------

creator_history["creator_prior_ctr"] = (
    creator_history["creator_prior_clicks"]
    / creator_history["creator_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

creator_history["creator_prior_engagement_rate"] = (
    creator_history["creator_prior_engagements"]
    / creator_history["creator_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

# ------------------------------------------
# Point-in-time smoothed creator rate
# ------------------------------------------

# Global engagement rate using ONLY historical data.
global_prior_engagement_rate = (
    history["meaningful_engagement"].sum()
    / len(history)
)

alpha = 10

creator_history["creator_prior_smoothed_engagement_rate"] = (
    creator_history["creator_prior_engagements"]
    + alpha * global_prior_engagement_rate
) / (
    creator_history["creator_prior_interactions"]
    + alpha
)

# ------------------------------------------
# Merge into candidate table
# ------------------------------------------

candidate_features = candidate_features.merge(
    creator_history,
    on="creator_id",
    how="left"
)

creator_behavior_columns = [
    "creator_prior_interactions",
    "creator_prior_clicks",
    "creator_prior_engagements",
    "creator_prior_ctr",
    "creator_prior_engagement_rate",
    "creator_prior_smoothed_engagement_rate"
]

candidate_features[creator_behavior_columns] = (
    candidate_features[creator_behavior_columns]
    .fillna(0)
)

# ------------------------------------------
# Verification
# ------------------------------------------

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print("\nCreator historical feature summary:")

display(
    candidate_features[
        ["creator_id"] + creator_behavior_columns
    ]
    .drop_duplicates()
    .head(10)
)

print("\nMissing creator features:")

print(
    candidate_features[creator_behavior_columns]
    .isna()
    .sum()
)

print(
    "\nHistorical global engagement rate:",
    round(global_prior_engagement_rate, 4)
)

print("\n✓ Creator historical features added")

Candidate feature table shape: (200, 32)

Creator historical feature summary:


,creator_id,creator_prior_interactions,creator_prior_clicks,creator_prior_engagements,creator_prior_ctr,creator_prior_engagement_rate,creator_prior_smoothed_engagement_rate
0,C0392,1039,374,373,0.3600,0.3590,0.3589
1,C0270,822,304,301,0.3698,0.3662,0.3660
3,C0350,303,99,99,0.3267,0.3267,0.3276
4,C0403,640,231,229,0.3609,0.3578,0.3577
5,C0318,790,289,285,0.3658,0.3608,0.3607
6,C0107,451,161,160,0.3570,0.3548,0.3547
9,C0492,667,242,241,0.3628,0.3613,0.3612
10,C0127,543,197,196,0.3628,0.3610,0.3608
11,C0286,348,117,116,0.3362,0.3333,0.3339
13,C0041,703,258,257,0.3670,0.3656,0.3654



Missing creator features:
creator_prior_interactions                0
creator_prior_clicks                      0
creator_prior_engagements                 0
creator_prior_ctr                         0
creator_prior_engagement_rate             0
creator_prior_smoothed_engagement_rate    0
dtype: int64

Historical global engagement rate: 0.3535

✓ Creator historical features added


In [228]:
# ==========================================
# ADD CONTENT HISTORICAL FEATURES
# ==========================================

content_history = (
    history
    .groupby("content_id")
    .agg(
        content_prior_interactions=("content_id", "size"),
        content_prior_clicks=("clicked", "sum"),
        content_prior_engagements=("meaningful_engagement", "sum")
    )
    .reset_index()
)

# ------------------------------------------
# Derived content rates
# ------------------------------------------

content_history["content_prior_ctr"] = (
    content_history["content_prior_clicks"]
    / content_history["content_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

content_history["content_prior_engagement_rate"] = (
    content_history["content_prior_engagements"]
    / content_history["content_prior_interactions"]
    .replace(0, np.nan)
).fillna(0)

# ------------------------------------------
# Point-in-time smoothed content rate
# ------------------------------------------

alpha = 10

content_history["content_prior_smoothed_engagement_rate"] = (
    content_history["content_prior_engagements"]
    + alpha * global_prior_engagement_rate
) / (
    content_history["content_prior_interactions"]
    + alpha
)

# ------------------------------------------
# Merge into candidate table
# ------------------------------------------

candidate_features = candidate_features.merge(
    content_history,
    on="content_id",
    how="left"
)

content_behavior_columns = [
    "content_prior_interactions",
    "content_prior_clicks",
    "content_prior_engagements",
    "content_prior_ctr",
    "content_prior_engagement_rate",
    "content_prior_smoothed_engagement_rate"
]

# New content may have no historical interactions.
candidate_features[content_behavior_columns] = (
    candidate_features[content_behavior_columns]
    .fillna(0)
)

# ------------------------------------------
# Verification
# ------------------------------------------

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print("\nContent historical feature summary:")

display(
    candidate_features[
        ["content_id"] + content_behavior_columns
    ].head(10)
)

print("\nMissing content features:")

print(
    candidate_features[content_behavior_columns]
    .isna()
    .sum()
)

print("\nNumber of candidates with zero history:")

print(
    (
        candidate_features["content_prior_interactions"] == 0
    ).sum()
)

print("\n✓ Content historical features added")

Candidate feature table shape: (200, 38)

Content historical feature summary:


,content_id,content_prior_interactions,content_prior_clicks,content_prior_engagements,content_prior_ctr,content_prior_engagement_rate,content_prior_smoothed_engagement_rate
0,CT005145,24,9,9,0.3750,0.3750,0.3687
1,CT004464,28,12,12,0.4286,0.4286,0.4088
2,CT001970,32,8,8,0.2500,0.2500,0.2746
3,CT001872,15,5,5,0.3333,0.3333,0.3414
4,CT009442,20,4,4,0.2000,0.2000,0.2512
5,CT003423,24,11,11,0.4583,0.4583,0.4275
6,CT002147,11,1,1,0.0909,0.0909,0.2160
7,CT008523,20,10,10,0.5000,0.5000,0.4512
8,CT003864,10,5,5,0.5000,0.5000,0.4267
9,CT009963,29,13,12,0.4483,0.4138,0.3983



Missing content features:
content_prior_interactions                0
content_prior_clicks                      0
content_prior_engagements                 0
content_prior_ctr                         0
content_prior_engagement_rate             0
content_prior_smoothed_engagement_rate    0
dtype: int64

Number of candidates with zero history:
0

✓ Content historical features added


In [229]:
# ==========================================
# ADD USER × CONTENT HISTORICAL FEATURES
# ==========================================

user_content_history = (
    history
    .groupby(
        ["user_id", "content_id"]
    )
    .agg(
        user_content_prior_interactions=(
            "content_id",
            "size"
        ),
        user_content_prior_clicks=(
            "clicked",
            "sum"
        ),
        user_content_prior_engagements=(
            "meaningful_engagement",
            "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------
# Derived user × content rates
# ------------------------------------------

user_content_history[
    "user_content_prior_ctr"
] = (
    user_content_history[
        "user_content_prior_clicks"
    ]
    /
    user_content_history[
        "user_content_prior_interactions"
    ].replace(0, np.nan)
).fillna(0)

user_content_history[
    "user_content_prior_engagement_rate"
] = (
    user_content_history[
        "user_content_prior_engagements"
    ]
    /
    user_content_history[
        "user_content_prior_interactions"
    ].replace(0, np.nan)
).fillna(0)

# ------------------------------------------
# Smoothed user × content engagement rate
# ------------------------------------------

alpha = 10

user_content_history[
    "user_content_prior_smoothed_engagement_rate"
] = (
    user_content_history[
        "user_content_prior_engagements"
    ]
    + alpha * global_prior_engagement_rate
) / (
    user_content_history[
        "user_content_prior_interactions"
    ]
    + alpha
)

# ------------------------------------------
# Merge into candidate table
# ------------------------------------------

candidate_features = candidate_features.merge(
    user_content_history,
    on=["user_id", "content_id"],
    how="left"
)

user_content_columns = [
    "user_content_prior_interactions",
    "user_content_prior_clicks",
    "user_content_prior_engagements",
    "user_content_prior_ctr",
    "user_content_prior_engagement_rate",
    "user_content_prior_smoothed_engagement_rate"
]

# Unseen user × content pairs have no history.
candidate_features[user_content_columns] = (
    candidate_features[user_content_columns]
    .fillna(0)
)

# ------------------------------------------
# Verification
# ------------------------------------------

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print(
    "\nUser × content history summary:"
)

display(
    candidate_features[
        ["user_id", "content_id"]
        + user_content_columns
    ].head(10)
)

print(
    "\nCandidates with prior user × content history:"
)

print(
    (
        candidate_features[
            "user_content_prior_interactions"
        ] > 0
    ).sum()
)

print(
    "\nMissing user × content features:"
)

print(
    candidate_features[user_content_columns]
    .isna()
    .sum()
)

print(
    "\n✓ User × content historical features added"
)

Candidate feature table shape: (200, 44)

User × content history summary:


,user_id,content_id,user_content_prior_interactions,user_content_prior_clicks,user_content_prior_engagements,user_content_prior_ctr,user_content_prior_engagement_rate,user_content_prior_smoothed_engagement_rate
0,U00001,CT005145,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,U00001,CT004464,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,U00001,CT001970,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,U00001,CT001872,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,U00001,CT009442,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
5,U00001,CT003423,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
6,U00001,CT002147,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,U00001,CT008523,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
8,U00001,CT003864,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
9,U00001,CT009963,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000



Candidates with prior user × content history:
0

Missing user × content features:
user_content_prior_interactions                0
user_content_prior_clicks                      0
user_content_prior_engagements                 0
user_content_prior_ctr                         0
user_content_prior_engagement_rate             0
user_content_prior_smoothed_engagement_rate    0
dtype: int64

✓ User × content historical features added


In [230]:
# ==========================================
# ADD USER × GENRE HISTORICAL FEATURES
# ==========================================

user_genre_history = (
    history
    .groupby(["user_id", "genre"])
    .agg(
        user_genre_prior_interactions=(
            "genre",
            "size"
        ),
        user_genre_prior_clicks=(
            "clicked",
            "sum"
        ),
        user_genre_prior_engagements=(
            "meaningful_engagement",
            "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------
# Derived user × genre engagement rate
# ------------------------------------------

user_genre_history[
    "user_genre_prior_engagement_rate"
] = (
    user_genre_history[
        "user_genre_prior_engagements"
    ]
    /
    user_genre_history[
        "user_genre_prior_interactions"
    ].replace(0, np.nan)
).fillna(0)

# ------------------------------------------
# Point-in-time smoothed rate
# ------------------------------------------

alpha = 10

user_genre_history[
    "user_genre_prior_smoothed_engagement_rate"
] = (
    user_genre_history[
        "user_genre_prior_engagements"
    ]
    + alpha * global_prior_engagement_rate
) / (
    user_genre_history[
        "user_genre_prior_interactions"
    ] + alpha
)

# ------------------------------------------
# Merge into candidate table
# ------------------------------------------

candidate_features = candidate_features.merge(
    user_genre_history,
    on=["user_id", "genre"],
    how="left"
)

user_genre_columns = [
    "user_genre_prior_interactions",
    "user_genre_prior_clicks",
    "user_genre_prior_engagements",
    "user_genre_prior_engagement_rate",
    "user_genre_prior_smoothed_engagement_rate"
]

candidate_features[user_genre_columns] = (
    candidate_features[user_genre_columns]
    .fillna(0)
)

# ------------------------------------------
# Verification
# ------------------------------------------

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print(
    "\nCandidates with prior history in their genre:"
)

print(
    (
        candidate_features[
            "user_genre_prior_interactions"
        ] > 0
    ).sum()
)

print(
    "\nMissing user × genre features:"
)

print(
    candidate_features[user_genre_columns]
    .isna()
    .sum()
)

print(
    "\n✓ User × genre historical features added"
)

Candidate feature table shape: (200, 49)

Candidates with prior history in their genre:
200

Missing user × genre features:
user_genre_prior_interactions                0
user_genre_prior_clicks                      0
user_genre_prior_engagements                 0
user_genre_prior_engagement_rate             0
user_genre_prior_smoothed_engagement_rate    0
dtype: int64

✓ User × genre historical features added


In [231]:
# ==========================================
# CHECK FEATURE CONTRACT
# ==========================================

built_columns = set(candidate_features.columns)
expected_columns = set(feature_columns)

missing_features = [
    col
    for col in feature_columns
    if col not in built_columns
]

extra_features = [
    col
    for col in candidate_features.columns
    if col not in feature_columns
]

print("Expected model features:", len(feature_columns))
print("Currently built columns:", len(candidate_features.columns))

print("\nMissing model features:")
for col in missing_features:
    print("-", col)

print("\nExtra columns:")
for col in extra_features:
    print("-", col)

print(
    "\nNumber of missing features:",
    len(missing_features)
)

print(
    "Number of extra columns:",
    len(extra_features)
)

Expected model features: 53
Currently built columns: 49

Missing model features:
- content_age_days
- freshness
- interaction_hour
- interaction_day_of_week
- is_weekend
- hour_sin
- hour_cos
- user_tenure_days
- user_prior_smoothed_engagement_rate

Extra columns:
- content_id
- creator_id
- user_id
- created_at
- signup_date

Number of missing features: 9
Number of extra columns: 5


In [232]:
# ==========================================
# ADD REMAINING POINT-IN-TIME FEATURES
# ==========================================

# ------------------------------------------
# 1. Content age at recommendation time
# ------------------------------------------

candidate_features["content_age_days"] = (
    cutoff
    - candidate_features["created_at"]
).dt.total_seconds() / (24 * 60 * 60)

assert (
    candidate_features["content_age_days"] >= 0
).all()

# ------------------------------------------
# 2. Freshness
# ------------------------------------------

candidate_features["freshness"] = np.exp(
    -candidate_features["content_age_days"] / 30
)

# ------------------------------------------
# 3. Recommendation-time context
# ------------------------------------------

candidate_features["interaction_hour"] = cutoff.hour

candidate_features["interaction_day_of_week"] = (
    cutoff.dayofweek
)

candidate_features["is_weekend"] = int(
    cutoff.dayofweek >= 5
)

# ------------------------------------------
# 4. Cyclical hour encoding
# ------------------------------------------

candidate_features["hour_sin"] = np.sin(
    2 * np.pi
    * candidate_features["interaction_hour"]
    / 24
)

candidate_features["hour_cos"] = np.cos(
    2 * np.pi
    * candidate_features["interaction_hour"]
    / 24
)

# ------------------------------------------
# 5. User tenure at recommendation time
# ------------------------------------------

candidate_features["user_tenure_days"] = (
    cutoff
    - candidate_features["signup_date"]
).dt.total_seconds() / (24 * 60 * 60)

assert (
    candidate_features["user_tenure_days"] >= 0
).all()

# ------------------------------------------
# 6. Point-in-time smoothed user rate
# ------------------------------------------

alpha = 10

candidate_features[
    "user_prior_smoothed_engagement_rate"
] = (
    candidate_features[
        "user_prior_engagements"
    ]
    + alpha * global_prior_engagement_rate
) / (
    candidate_features[
        "user_prior_interactions"
    ]
    + alpha
)

# ------------------------------------------
# Verification
# ------------------------------------------

remaining_features = [
    "content_age_days",
    "freshness",
    "interaction_hour",
    "interaction_day_of_week",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "user_tenure_days",
    "user_prior_smoothed_engagement_rate"
]

print(
    "Candidate feature table shape:",
    candidate_features.shape
)

print("\nRemaining features:")

display(
    candidate_features[
        ["content_id"] + remaining_features
    ].head(10)
)

print("\nMissing values:")

print(
    candidate_features[remaining_features]
    .isna()
    .sum()
)

print("\n✓ Remaining point-in-time features added")

Candidate feature table shape: (200, 58)

Remaining features:


,content_id,content_age_days,freshness,interaction_hour,interaction_day_of_week,is_weekend,hour_sin,hour_cos,user_tenure_days,user_prior_smoothed_engagement_rate
0,CT005145,176.9997,0.0027,5,5,1,0.9659,0.2588,918.4058,0.3796
1,CT004464,219.8425,0.0007,5,5,1,0.9659,0.2588,918.4058,0.3796
2,CT001970,368.2582,0.0000,5,5,1,0.9659,0.2588,918.4058,0.3796
3,CT001872,192.9864,0.0016,5,5,1,0.9659,0.2588,918.4058,0.3796
4,CT009442,193.4058,0.0016,5,5,1,0.9659,0.2588,918.4058,0.3796
5,CT003423,267.5597,0.0001,5,5,1,0.9659,0.2588,918.4058,0.3796
6,CT002147,361.8608,0.0000,5,5,1,0.9659,0.2588,918.4058,0.3796
7,CT008523,385.8023,0.0000,5,5,1,0.9659,0.2588,918.4058,0.3796
8,CT003864,189.8873,0.0018,5,5,1,0.9659,0.2588,918.4058,0.3796
9,CT009963,356.3845,0.0000,5,5,1,0.9659,0.2588,918.4058,0.3796



Missing values:
content_age_days                       0
freshness                              0
interaction_hour                       0
interaction_day_of_week                0
is_weekend                             0
hour_sin                               0
hour_cos                               0
user_tenure_days                       0
user_prior_smoothed_engagement_rate    0
dtype: int64

✓ Remaining point-in-time features added


In [233]:
# ==========================================
# FINAL FEATURE CONTRACT CHECK
# ==========================================

model_features = candidate_features[
    feature_columns
].copy()

print("Expected model features:", len(feature_columns))
print("Actual model features:", model_features.shape[1])
print("Candidate rows:", model_features.shape[0])

# Exact column order
assert model_features.columns.tolist() == feature_columns

# No missing values
assert model_features.isna().sum().sum() == 0

# Correct number of rows
assert len(model_features) == len(candidate_features)

# All required features present
assert set(feature_columns) == set(model_features.columns)

print("\n✓ Exactly 53 model features present")
print("✓ Correct feature order")
print("✓ No missing values")
print("✓ 200 candidate rows preserved")
print("\n✓ MODEL FEATURE CONTRACT VALIDATED")

Expected model features: 53
Actual model features: 53
Candidate rows: 200

✓ Exactly 53 model features present
✓ Correct feature order
✓ No missing values
✓ 200 candidate rows preserved

✓ MODEL FEATURE CONTRACT VALIDATED


In [234]:
# ==========================================
# ATTACH CONTENT-BASED SIMILARITY SCORES
# ==========================================

# Keep only the columns we need from the original
# content-based candidate set.
similarity_lookup = (
    ml_candidates[
        ["content_id", "similarity_score"]
    ]
    .drop_duplicates("content_id")
)

# Attach the similarity score to the ML candidate table.
candidate_features = candidate_features.merge(
    similarity_lookup,
    on="content_id",
    how="left",
    validate="one_to_one"
)

# Verify that the merge preserved all candidates.
assert len(candidate_features) == 200

# Every candidate should have a similarity score because
# candidate_features came from ml_candidates.
assert candidate_features["similarity_score"].notna().all()

print("Candidate rows:", len(candidate_features))
print(
    "Missing similarity scores:",
    candidate_features["similarity_score"].isna().sum()
)

print("\n✓ Similarity scores attached")
print("✓ 200 candidate rows preserved")

Candidate rows: 200
Missing similarity scores: 0

✓ Similarity scores attached
✓ 200 candidate rows preserved


In [235]:
# ==========================================
# SCORE RECOMMENDATION CANDIDATES
# ==========================================

candidate_features["engagement_score"] = (
    random_forest.predict_proba(model_features)[:, 1]
)

ml_ranked_candidates = (
    candidate_features
    .sort_values(
        "engagement_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ml_ranked_candidates["ml_rank"] = (
    np.arange(len(ml_ranked_candidates)) + 1
)

display_columns = [
    "ml_rank",
    "content_id",
    "creator_id",
    "genre",
    "content_type",
    "duration",
    "creator_followers",
    "content_age_days",
    "freshness",
    "similarity_score",
    "engagement_score"
]

print("Top 20 ML-ranked recommendations:")

display(
    ml_ranked_candidates[
        display_columns
    ].head(20)
)

print("\nScore summary:")

print(
    ml_ranked_candidates[
        "engagement_score"
    ].describe()
)

print("\n✓ Engagement scores generated")
print("✓ Candidates ranked by predicted engagement")

Top 20 ML-ranked recommendations:


,ml_rank,content_id,creator_id,genre,content_type,duration,creator_followers,content_age_days,freshness,similarity_score,engagement_score
0,1,CT001488,C0392,Action,mini,17.4000,321,22.6402,0.4702,0.3068,0.6114
1,2,CT001523,C0036,Comedy,mini,9.7000,119,9.6306,0.7254,0.3378,0.5908
2,3,CT006338,C0177,Action,cine,25.1000,78,50.4092,0.1863,0.3227,0.5785
3,4,CT004866,C0270,Action,mini,15.1000,64,38.9677,0.2728,0.3049,0.5762
4,5,CT006944,C0350,Action,mini,8.0000,10,1.2261,0.9600,0.3334,0.5745
5,6,CT009762,C0041,Comedy,mini,7.7000,170,63.3395,0.1211,0.3326,0.5717
6,7,CT006446,C0067,Action,mini,18.3000,71,52.1075,0.1761,0.3000,0.5655
7,8,CT002709,C0270,Action,video,66.7000,64,78.8571,0.0722,0.3116,0.5617
8,9,CT008295,C0270,Comedy,mini,14.6000,64,145.8827,0.0077,0.3057,0.5579
9,10,CT004288,C0177,Action,mini,14.5000,78,121.2993,0.0175,0.3004,0.5574



Score summary:
count   200.0000
mean      0.5090
std       0.0302
min       0.4287
25%       0.4940
50%       0.5103
75%       0.5266
max       0.6114
Name: engagement_score, dtype: float64

✓ Engagement scores generated
✓ Candidates ranked by predicted engagement


In [236]:
# ==========================================
# COMPARE CONTENT-BASED VS ML RANKING
# ==========================================

comparison = (
    ml_ranked_candidates[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration",
            "similarity_score",
            "engagement_score"
        ]
    ]
    .copy()
)

# Original content-based rank
comparison["tfidf_rank"] = (
    comparison["similarity_score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)

# Sort by ML ranking
comparison = (
    comparison
    .sort_values("engagement_score", ascending=False)
    .reset_index(drop=True)
)

comparison["ml_rank"] = (
    np.arange(len(comparison)) + 1
)

# How much did the ML model move each item?
comparison["rank_change"] = (
    comparison["tfidf_rank"]
    - comparison["ml_rank"]
)

display(
    comparison[
        [
            "ml_rank",
            "tfidf_rank",
            "rank_change",
            "content_id",
            "creator_id",
            "genre",
            "similarity_score",
            "engagement_score"
        ]
    ].head(20)
)

print("\nRanking correlation:")

print(
    comparison[
        ["tfidf_rank", "ml_rank"]
    ].corr(method="spearman").iloc[0, 1]
)

print("\nAverage absolute rank change:")

print(
    comparison["rank_change"]
    .abs()
    .mean()
)

print("\n✓ Content-based and ML rankings compared")

,ml_rank,tfidf_rank,rank_change,content_id,creator_id,genre,similarity_score,engagement_score
0,1,116,115,CT001488,C0392,Action,0.3068,0.6114
1,2,22,20,CT001523,C0036,Comedy,0.3378,0.5908
2,3,76,73,CT006338,C0177,Action,0.3227,0.5785
3,4,136,132,CT004866,C0270,Action,0.3049,0.5762
4,5,41,36,CT006944,C0350,Action,0.3334,0.5745
5,6,44,38,CT009762,C0041,Comedy,0.3326,0.5717
6,7,167,160,CT006446,C0067,Action,0.3000,0.5655
7,8,98,90,CT002709,C0270,Action,0.3116,0.5617
8,9,129,120,CT008295,C0270,Comedy,0.3057,0.5579
9,10,165,155,CT004288,C0177,Action,0.3004,0.5574



Ranking correlation:
0.20506064517182146

Average absolute rank change:
58.8

✓ Content-based and ML rankings compared


In [237]:
# ==========================================
# INSPECT FINAL RANKING COMPONENTS
# ==========================================

ranking_preview = ml_ranked_candidates[
    [
        "content_id",
        "creator_id",
        "genre",
        "content_type",
        "duration",
        "similarity_score",
        "engagement_score",
        "freshness",
        "creator_followers"
    ]
].copy()

# Normalize creator popularity using log scale.
# This prevents very large creators from dominating.
ranking_preview["creator_popularity"] = np.log1p(
    ranking_preview["creator_followers"]
)

# Min-max normalization helper
def min_max_scale(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            0.5,
            index=series.index
        )

    return (
        (series - min_value)
        / (max_value - min_value)
    )

ranking_preview["engagement_score_norm"] = (
    min_max_scale(
        ranking_preview["engagement_score"]
    )
)

ranking_preview["freshness_norm"] = (
    min_max_scale(
        ranking_preview["freshness"]
    )
)

ranking_preview["popularity_norm"] = (
    min_max_scale(
        ranking_preview["creator_popularity"]
    )
)

display(
    ranking_preview.head(10)
)

print("\n✓ Ranking components prepared")

,content_id,creator_id,genre,content_type,duration,similarity_score,engagement_score,freshness,creator_followers,creator_popularity,engagement_score_norm,freshness_norm,popularity_norm
0,CT001488,C0392,Action,mini,17.4000,0.3068,0.6114,0.4702,321,5.7746,1.0000,0.4898,1.0000
1,CT001523,C0036,Comedy,mini,9.7000,0.3378,0.5908,0.7254,119,4.7875,0.8870,0.7557,0.7077
2,CT006338,C0177,Action,cine,25.1000,0.3227,0.5785,0.1863,78,4.3694,0.8199,0.1941,0.5839
3,CT004866,C0270,Action,mini,15.1000,0.3049,0.5762,0.2728,64,4.1744,0.8072,0.2842,0.5261
4,CT006944,C0350,Action,mini,8.0000,0.3334,0.5745,0.9600,10,2.3979,0.7981,1.0000,0.0000
5,CT009762,C0041,Comedy,mini,7.7000,0.3326,0.5717,0.1211,170,5.1417,0.7823,0.1261,0.8126
6,CT006446,C0067,Action,mini,18.3000,0.3000,0.5655,0.1761,71,4.2767,0.7483,0.1834,0.5564
7,CT002709,C0270,Action,video,66.7000,0.3116,0.5617,0.0722,64,4.1744,0.7279,0.0752,0.5261
8,CT008295,C0270,Comedy,mini,14.6000,0.3057,0.5579,0.0077,64,4.1744,0.7069,0.0081,0.5261
9,CT004288,C0177,Action,mini,14.5000,0.3004,0.5574,0.0175,78,4.3694,0.7043,0.0183,0.5839



✓ Ranking components prepared


In [238]:
# ==========================================
# FINAL FEED RANKER
# ==========================================

def build_final_feed(
    candidates,
    k=10,
    engagement_weight=0.75,
    freshness_weight=0.15,
    exploration_weight=0.10,
    creator_penalty=0.10,
    genre_penalty=0.05
):
    """
    Build a diversified recommendation feed.

    The Random Forest engagement score provides the
    primary ranking signal.

    Freshness adds recency preference.

    Exploration gives a small opportunity to surface
    candidates that are not simply the highest-scoring
    items.

    Creator and genre penalties encourage diversity.
    """

    df = candidates.copy()

    # ------------------------------------------
    # Normalize ranking signals
    # ------------------------------------------

    df["engagement_norm"] = min_max_scale(
        df["engagement_score"]
    )

    df["freshness_norm"] = min_max_scale(
        df["freshness"]
    )

    # Random but reproducible exploration signal.
    # This is intentionally small.
    rng = np.random.default_rng(42)

    df["exploration_score"] = (
        rng.random(len(df))
    )

    # ------------------------------------------
    # Base ranking score
    # ------------------------------------------

    df["base_score"] = (
        engagement_weight * df["engagement_norm"]
        + freshness_weight * df["freshness_norm"]
        + exploration_weight * df["exploration_score"]
    )

    # ------------------------------------------
    # Greedy diversified selection
    # ------------------------------------------

    remaining = df.copy()

    selected_rows = []

    creator_counts = {}
    genre_counts = {}

    for _ in range(min(k, len(remaining))):

        if len(remaining) == 0:
            break

        scores = remaining["base_score"].copy()

        # Penalize creators already appearing in feed
        for idx in remaining.index:

            creator = remaining.loc[
                idx, "creator_id"
            ]

            genre = remaining.loc[
                idx, "genre"
            ]

            creator_penalty_value = (
                creator_penalty
                * creator_counts.get(
                    creator, 0
                )
            )

            genre_penalty_value = (
                genre_penalty
                * genre_counts.get(
                    genre, 0
                )
            )

            scores.loc[idx] -= (
                creator_penalty_value
                + genre_penalty_value
            )

        # Pick best remaining candidate
        best_idx = scores.idxmax()

        selected = remaining.loc[
            best_idx
        ].copy()

        selected["final_score"] = (
            scores.loc[best_idx]
        )

        selected["feed_position"] = (
            len(selected_rows) + 1
        )

        selected_rows.append(selected)

        # Update diversity counts
        creator = selected["creator_id"]
        genre = selected["genre"]

        creator_counts[creator] = (
            creator_counts.get(creator, 0) + 1
        )

        genre_counts[genre] = (
            genre_counts.get(genre, 0) + 1
        )

        # Remove selected candidate
        remaining = remaining.drop(
            best_idx
        )

    final_feed = pd.DataFrame(
        selected_rows
    ).reset_index(drop=True)

    return final_feed

In [239]:
# ==========================================
# GENERATE FINAL TOP-10 FEED
# ==========================================

final_feed = build_final_feed(
    ml_ranked_candidates,
    k=10
)

feed_columns = [
    "feed_position",
    "content_id",
    "creator_id",
    "genre",
    "content_type",
    "duration",
    "engagement_score",
    "freshness",
    "similarity_score",
    "final_score"
]

display(
    final_feed[
        feed_columns
    ]
)

print("\nCreator distribution:")
print(
    final_feed["creator_id"]
    .value_counts()
)

print("\nGenre distribution:")
print(
    final_feed["genre"]
    .value_counts()
)

print(
    "\n✓ Final diversified Top-10 feed generated"
)

,feed_position,content_id,creator_id,genre,content_type,duration,engagement_score,freshness,similarity_score,final_score
0,1,CT001488,C0392,Action,mini,17.4000,0.6114,0.4702,0.3068,0.9009
1,2,CT001523,C0036,Comedy,mini,9.7000,0.5908,0.7254,0.3378,0.8225
2,3,CT006944,C0350,Action,mini,8.0000,0.5745,0.9600,0.3334,0.7080
3,4,CT009762,C0041,Comedy,mini,7.7000,0.5717,0.1211,0.3326,0.6532
4,5,CT000581,C0127,Documentary,image,91.0000,0.5420,0.4988,0.3168,0.6398
5,6,CT006338,C0177,Action,cine,25.1000,0.5785,0.1863,0.3227,0.6299
6,7,CT004866,C0270,Action,mini,15.1000,0.5762,0.2728,0.3049,0.5677
7,8,CT007823,C0490,Fantasy,image,29.9000,0.5232,0.8272,0.3320,0.5630
8,9,CT006741,C0107,Drama,mini,6.9000,0.5267,0.5438,0.3044,0.5552
9,10,CT007433,C0420,Documentary,mini,13.9000,0.5320,0.4906,0.3024,0.5341



Creator distribution:
creator_id
C0392    1
C0036    1
C0350    1
C0041    1
C0127    1
C0177    1
C0270    1
C0490    1
C0107    1
C0420    1
Name: count, dtype: int64

Genre distribution:
genre
Action         4
Comedy         2
Documentary    2
Fantasy        1
Drama          1
Name: count, dtype: int64

✓ Final diversified Top-10 feed generated


In [240]:
# ==========================================
# RECOMMENDATION EVALUATION METRICS
# ==========================================

def precision_at_k(recommended, relevant, k):
    """
    Precision@K:
    Fraction of recommended items in the top K
    that were actually relevant.
    """
    recommended = recommended[:k]

    if len(recommended) == 0:
        return 0.0

    hits = len(
        set(recommended) & set(relevant)
    )

    return hits / len(recommended)


def recall_at_k(recommended, relevant, k):
    """
    Recall@K:
    Fraction of relevant items retrieved
    in the top K recommendations.
    """
    recommended = recommended[:k]

    if len(relevant) == 0:
        return 0.0

    hits = len(
        set(recommended) & set(relevant)
    )

    return hits / len(relevant)


def average_precision_at_k(
    recommended,
    relevant,
    k
):
    """
    Average Precision@K.
    Rewards relevant items appearing earlier
    in the recommendation list.
    """

    recommended = recommended[:k]
    relevant = set(relevant)

    if len(relevant) == 0:
        return 0.0

    score = 0.0
    hits = 0

    for rank, item in enumerate(
        recommended,
        start=1
    ):

        if item in relevant:

            hits += 1

            score += (
                hits / rank
            )

    return score / min(
        len(relevant),
        k
    )


def ndcg_at_k(
    recommended,
    relevant,
    k
):
    """
    NDCG@K for binary relevance.
    """

    recommended = recommended[:k]
    relevant = set(relevant)

    dcg = 0.0

    for rank, item in enumerate(
        recommended,
        start=1
    ):

        if item in relevant:
            dcg += (
                1 /
                np.log2(rank + 1)
            )

    ideal_hits = min(
        len(relevant),
        k
    )

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1 /
        np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    return dcg / idcg

In [241]:
# ==========================================
# TEST RECOMMENDATION METRICS
# ==========================================

recommended = [
    "A",
    "B",
    "C",
    "D",
    "E"
]

relevant = [
    "A",
    "C",
    "F"
]

print(
    "Precision@5:",
    precision_at_k(
        recommended,
        relevant,
        5
    )
)

print(
    "Recall@5:",
    recall_at_k(
        recommended,
        relevant,
        5
    )
)

print(
    "MAP@5:",
    average_precision_at_k(
        recommended,
        relevant,
        5
    )
)

print(
    "NDCG@5:",
    ndcg_at_k(
        recommended,
        relevant,
        5
    )
)

print("\n✓ Recommendation metrics validated")

Precision@5: 0.4
Recall@5: 0.6666666666666666
MAP@5: 0.5555555555555555
NDCG@5: 0.7039180890341347

✓ Recommendation metrics validated


In [242]:
# ==========================================
# BUILD OFFLINE EVALUATION UNIVERSE
# ==========================================

# Future interactions are deliberately hidden
# from the recommendation generation process.
future_interactions = interactions[
    interactions["timestamp"] >= cutoff
].copy()

# Relevant future content:
# content that received meaningful engagement
# after the recommendation cutoff.
future_positive = future_interactions[
    future_interactions["meaningful_engagement"] == 1
].copy()

# Ground-truth relevant items for each user.
ground_truth = (
    future_positive
    .groupby("user_id")["content_id"]
    .apply(set)
    .to_dict()
)

# Users with at least one future positive interaction.
evaluation_users = sorted(
    ground_truth.keys()
)

print(
    "Future interactions:",
    len(future_interactions)
)

print(
    "Future positive interactions:",
    len(future_positive)
)

print(
    "Evaluation users:",
    len(evaluation_users)
)

print(
    "Average relevant items per user:",
    np.mean([
        len(items)
        for items in ground_truth.values()
    ])
)

print(
    "Median relevant items per user:",
    np.median([
        len(items)
        for items in ground_truth.values()
    ])
)

print(
    "\n✓ Evaluation universe created"
)

Future interactions: 37500
Future positive interactions: 13863
Evaluation users: 4015
Average relevant items per user: 3.4493150684931506
Median relevant items per user: 2.0

✓ Evaluation universe created


In [243]:
# ==========================================
# BASELINE RECOMMENDERS FOR EVALUATION
# ==========================================

K = 10

# ------------------------------------------
# Global popularity
# ------------------------------------------

global_popularity = (
    historical_interactions
    .groupby("content_id")
    .agg(
        engagement_count=(
            "meaningful_engagement",
            "sum"
        ),
        interaction_count=(
            "content_id",
            "size"
        ),
        watch_time=(
            "watch_time",
            "sum"
        )
    )
    .reset_index()
)

global_popularity["engagement_rate"] = (
    global_popularity["engagement_count"]
    / global_popularity["interaction_count"]
)

global_popularity = (
    global_popularity
    .merge(
        eligible_content[
            [
                "content_id",
                "creator_id",
                "genre",
                "created_at"
            ]
        ],
        on="content_id",
        how="left"
    )
    .sort_values(
        [
            "engagement_count",
            "engagement_rate"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------
# Recent popularity
# ------------------------------------------

recent_window_start = (
    cutoff - pd.Timedelta(days=30)
)

recent_interactions = historical_interactions[
    historical_interactions["timestamp"]
    >= recent_window_start
].copy()

recent_popularity = (
    recent_interactions
    .groupby("content_id")
    .agg(
        engagement_count=(
            "meaningful_engagement",
            "sum"
        ),
        interaction_count=(
            "content_id",
            "size"
        )
    )
    .reset_index()
)

recent_popularity["engagement_rate"] = (
    recent_popularity["engagement_count"]
    / recent_popularity["interaction_count"]
)

recent_popularity = (
    recent_popularity
    .merge(
        eligible_content[
            [
                "content_id",
                "creator_id",
                "genre",
                "created_at"
            ]
        ],
        on="content_id",
        how="left"
    )
    .sort_values(
        [
            "engagement_count",
            "engagement_rate"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# ------------------------------------------
# Recommendation functions
# ------------------------------------------

def recommend_global_popularity(
    user_id,
    k=10
):
    seen = seen_content_by_user.get(
        user_id,
        set()
    )

    recommendations = (
        global_popularity[
            ~global_popularity["content_id"]
            .isin(seen)
        ]
        .head(k)
        ["content_id"]
        .tolist()
    )

    return recommendations


def recommend_recent_popularity(
    user_id,
    k=10
):
    seen = seen_content_by_user.get(
        user_id,
        set()
    )

    recommendations = (
        recent_popularity[
            ~recent_popularity["content_id"]
            .isin(seen)
        ]
        .head(k)
        ["content_id"]
        .tolist()
    )

    return recommendations


print(
    "Global popularity candidates:",
    len(global_popularity)
)

print(
    "Recent popularity candidates:",
    len(recent_popularity)
)

print(
    "Recent window starts:",
    recent_window_start
)

print(
    "\n✓ Global popularity recommender ready"
)

print(
    "✓ Recent popularity recommender ready"
)

Global popularity candidates: 9638
Recent popularity candidates: 8907
Recent window starts: 2026-07-16 05:05:33.349999872

✓ Global popularity recommender ready
✓ Recent popularity recommender ready


In [244]:
# ==========================================
# CONTENT-BASED RECOMMENDER
# ==========================================

def recommend_content_based(
    user_id,
    k=10
):
    """
    Recommend unseen content based on the user's
    historical positive-engagement profile.

    Uses the TF-IDF representation created earlier.
    """

    # ------------------------------------------
    # User's positive historical interactions
    # ------------------------------------------

    user_positive = historical_interactions[
        (historical_interactions["user_id"] == user_id)
        & (
            historical_interactions[
                "meaningful_engagement"
            ] == 1
        )
    ]

    positive_content_ids = (
        user_positive[
            "content_id"
        ]
        .tolist()
    )

    # If the user has no positive history,
    # return an empty list.
    if len(positive_content_ids) == 0:
        return []

    # ------------------------------------------
    # Get TF-IDF vectors for positive content
    # ------------------------------------------

    positive_indices = []

    for content_id in positive_content_ids:

        if content_id in content_id_to_index.index:

            positive_indices.append(
                content_id_to_index[
                    content_id
                ]
            )

    if len(positive_indices) == 0:
        return []

    # ------------------------------------------
    # Build user profile
    # ------------------------------------------

    user_profile = (
        content_matrix[
            positive_indices
        ]
        .mean(axis=0)
    )

    # Convert to sparse matrix
    user_profile = csr_matrix(
        user_profile
    )

    # ------------------------------------------
    # Calculate similarity
    # ------------------------------------------

    similarities = (
        cosine_similarity(
            user_profile,
            content_matrix
        )
        .ravel()
    )

    similarity_df = pd.DataFrame({
        "content_id":
            content_features[
                "content_id"
            ].values,
        "similarity_score":
            similarities
    })

    # ------------------------------------------
    # Remove already-seen content
    # ------------------------------------------

    seen = seen_content_by_user.get(
        user_id,
        set()
    )

    similarity_df = (
        similarity_df[
            ~similarity_df[
                "content_id"
            ].isin(seen)
        ]
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .head(k)
    )

    return (
        similarity_df[
            "content_id"
        ]
        .tolist()
    )


print("✓ Content-based recommender ready")

✓ Content-based recommender ready


In [245]:
from scipy.sparse import csr_matrix

print("✓ csr_matrix imported successfully")

✓ csr_matrix imported successfully


In [246]:
# ==========================================
# BUILD USER → SEEN CONTENT LOOKUP
# ==========================================

seen_content_by_user = (
    historical_interactions
    .groupby("user_id")["content_id"]
    .apply(set)
    .to_dict()
)

print(
    "Users with historical interactions:",
    len(seen_content_by_user)
)

print(
    "Example user:",
    list(seen_content_by_user.keys())[0]
)

print(
    "Example seen-content count:",
    len(
        seen_content_by_user[
            list(seen_content_by_user.keys())[0]
        ]
    )
)

print("✓ Seen-content lookup created")

Users with historical interactions: 4998
Example user: U00001
Example seen-content count: 51
✓ Seen-content lookup created


In [247]:
# ==========================================
# TEST CONTENT-BASED RECOMMENDER
# ==========================================

content_recs = recommend_content_based(
    test_user,
    K
)

print("Test user:", test_user)

print("\nContent-based recommendations:")
print(content_recs)

print("\nNumber of recommendations:", len(content_recs))

print("\nSeen-content overlap:")
print(
    len(
        set(content_recs)
        &
        seen_content_by_user.get(
            test_user,
            set()
        )
    )
)

print("\n✓ Content-based recommender tested")

Test user: U00001

Content-based recommendations:
['CT005145', 'CT004464', 'CT001970', 'CT001872', 'CT009442', 'CT003423', 'CT008523', 'CT002147', 'CT003864', 'CT009963']

Number of recommendations: 10

Seen-content overlap:
0

✓ Content-based recommender tested


In [248]:
# ==========================================
# BUILD FAIR OFFLINE EVALUATION GROUND TRUTH
# ==========================================

# Content available at recommendation time
eligible_content_ids = set(
    eligible_content["content_id"]
)

# Future meaningful interactions
future_positive = future_interactions[
    future_interactions["meaningful_engagement"] == 1
].copy()

# Keep only content that:
# 1. existed by the cutoff
# 2. was NOT already seen by the user before the cutoff

def is_valid_ground_truth(row):
    user_id = row["user_id"]
    content_id = row["content_id"]

    seen_items = seen_content_by_user.get(
        user_id,
        set()
    )

    return (
        content_id in eligible_content_ids
        and content_id not in seen_items
    )


future_positive["valid_for_evaluation"] = (
    future_positive.apply(
        is_valid_ground_truth,
        axis=1
    )
)

fair_future_positive = future_positive[
    future_positive["valid_for_evaluation"]
].copy()

# Build user → relevant content mapping
ground_truth = (
    fair_future_positive
    .groupby("user_id")["content_id"]
    .apply(set)
    .to_dict()
)

# Only users with at least one valid future positive
evaluation_users = sorted(
    ground_truth.keys()
)

print("Future positive interactions:", len(future_positive))
print(
    "Fair evaluation positives:",
    len(fair_future_positive)
)
print(
    "Evaluation users:",
    len(evaluation_users)
)

relevant_counts = pd.Series(
    [len(v) for v in ground_truth.values()]
)

print(
    "Average relevant items/user:",
    round(relevant_counts.mean(), 3)
)

print(
    "Median relevant items/user:",
    relevant_counts.median()
)

print("\n✓ Fair offline evaluation ground truth created")

Future positive interactions: 13863
Fair evaluation positives: 10295
Evaluation users: 3642
Average relevant items/user: 2.824
Median relevant items/user: 2.0

✓ Fair offline evaluation ground truth created


In [249]:
# ==========================================
# EVALUATE POPULARITY BASELINES
# ==========================================

K = 10


def evaluate_recommender(
    recommend_function,
    users,
    ground_truth,
    k=10
):
    """
    Evaluate a recommender using macro-averaged
    Precision@K, Recall@K, MAP@K and NDCG@K.
    """

    precision_scores = []
    recall_scores = []
    map_scores = []
    ndcg_scores = []

    evaluated_users = 0

    for user_id in users:

        relevant = ground_truth.get(user_id, set())

        if len(relevant) == 0:
            continue

        recommendations = recommend_function(
            user_id,
            k
        )

        precision_scores.append(
            precision_at_k(
                recommendations,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommendations,
                relevant,
                k
            )
        )

        map_scores.append(
            average_precision_at_k(
                recommendations,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommendations,
                relevant,
                k
            )
        )

        evaluated_users += 1

    return {
        "Precision@10": np.mean(precision_scores),
        "Recall@10": np.mean(recall_scores),
        "MAP@10": np.mean(map_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "Users evaluated": evaluated_users
    }


# ------------------------------------------
# Global popularity
# ------------------------------------------

print("Evaluating global popularity...")

global_pop_results = evaluate_recommender(
    recommend_global_popularity,
    evaluation_users,
    ground_truth,
    K
)


# ------------------------------------------
# Recent popularity
# ------------------------------------------

print("Evaluating recent popularity...")

recent_pop_results = evaluate_recommender(
    recommend_recent_popularity,
    evaluation_users,
    ground_truth,
    K
)


# ------------------------------------------
# Results
# ------------------------------------------

popularity_results = pd.DataFrame([
    {
        "Strategy": "Global Popularity",
        **global_pop_results
    },
    {
        "Strategy": "Recent Popularity",
        **recent_pop_results
    }
])

print("\n==========================================")
print("POPULARITY BASELINE RESULTS")
print("==========================================")

display(
    popularity_results.round(4)
)

Evaluating global popularity...
Evaluating recent popularity...

POPULARITY BASELINE RESULTS


,Strategy,Precision@10,Recall@10,MAP@10,NDCG@10,Users evaluated
0,Global Popularity,0.0002,0.0007,0.0002,0.0005,3642
1,Recent Popularity,0.0014,0.0047,0.0018,0.0032,3642


In [250]:
# ==========================================
# EVALUATE CONTENT-BASED RECOMMENDER
# ==========================================

print("Evaluating content-based recommender...")

content_based_results = evaluate_recommender(
    recommend_content_based,
    evaluation_users,
    ground_truth,
    K
)

content_based_results_df = pd.DataFrame([
    {
        "Strategy": "Content-Based",
        **content_based_results
    }
])

print("\n==========================================")
print("CONTENT-BASED RESULTS")
print("==========================================")

print(
    content_based_results_df.to_string(
        index=False
    )
)

Evaluating content-based recommender...

CONTENT-BASED RESULTS
     Strategy  Precision@10  Recall@10  MAP@10  NDCG@10  Users evaluated
Content-Based        0.0005     0.0016  0.0005   0.0010             3642


In [251]:
# ==========================================
# GROUND-TRUTH DIAGNOSTIC
# ==========================================

# Number of eligible unseen content items per user
candidate_counts = []

# Number of relevant future items that each user has
relevant_counts = []

for user_id in evaluation_users:

    seen = seen_content_by_user.get(
        user_id,
        set()
    )

    unseen_eligible = (
        eligible_content_ids - seen
    )

    candidate_counts.append(
        len(unseen_eligible)
    )

    relevant_counts.append(
        len(ground_truth[user_id])
    )

candidate_counts = pd.Series(candidate_counts)
relevant_counts = pd.Series(relevant_counts)

print("==========================================")
print("EVALUATION DIAGNOSTIC")
print("==========================================")

print(
    "\nEligible unseen candidates per user:"
)

print(
    candidate_counts.describe().round(2)
)

print(
    "\nRelevant future items per user:"
)

print(
    relevant_counts.describe().round(2)
)

print(
    "\nUsers with exactly 1 relevant item:",
    (relevant_counts == 1).sum()
)

print(
    "Users with 2+ relevant items:",
    (relevant_counts >= 2).sum()
)

print(
    "Users with 5+ relevant items:",
    (relevant_counts >= 5).sum()
)

print(
    "\n✓ Evaluation diagnostic complete"
)

EVALUATION DIAGNOSTIC

Eligible unseen candidates per user:
count   3,642.0000
mean    9,608.3800
std        43.5200
min     9,175.0000
25%     9,597.0000
50%     9,620.0000
75%     9,635.0000
max     9,658.0000
dtype: float64

Relevant future items per user:
count   3,642.0000
mean        2.8200
std         2.4800
min         1.0000
25%         1.0000
50%         2.0000
75%         4.0000
max        30.0000
dtype: float64

Users with exactly 1 relevant item: 1279
Users with 2+ relevant items: 2363
Users with 5+ relevant items: 593

✓ Evaluation diagnostic complete


In [252]:
# ==========================================
# PERSONALIZED ML RECOMMENDER
# ==========================================

ML_CANDIDATE_K = 200


def recommend_ml(
    user_id,
    k=10,
    candidate_k=ML_CANDIDATE_K
):
    """
    Personalized ML recommender.

    Pipeline:
        User
          ↓
        Content-based candidate generation
          ↓
        Seen-content filtering
          ↓
        Point-in-time feature construction
          ↓
        Random Forest engagement scoring
          ↓
        Engagement-based ranking

    The function reproduces the 53-feature model contract
    used during model training.
    """

    # ======================================
    # STEP 1: GENERATE CANDIDATES
    # ======================================

    candidate_ids = recommend_content_based(
        user_id,
        candidate_k
    )

    if len(candidate_ids) == 0:
        return []

    candidates = content_features[
        content_features["content_id"].isin(candidate_ids)
    ].copy()

    if len(candidates) == 0:
        return []


    # ======================================
    # STEP 2: USER HISTORY
    # ======================================

    user_history = historical_interactions[
        historical_interactions["user_id"] == user_id
    ].copy()

    user_row = users[
        users["user_id"] == user_id
    ].iloc[0]

    user_interactions = len(user_history)

    user_clicks = user_history["clicked"].sum()

    user_engagements = (
        user_history["meaningful_engagement"].sum()
    )

    user_watch_time = (
        user_history["watch_time"].sum()
    )

    user_completions = (
        user_history["completion_rate"].sum()
    )

    user_ctr = (
        user_clicks / user_interactions
        if user_interactions > 0
        else 0.0
    )

    user_engagement_rate = (
        user_engagements / user_interactions
        if user_interactions > 0
        else 0.0
    )

    user_avg_watch = (
        user_history["watch_time"].mean()
        if user_interactions > 0
        else 0.0
    )

    user_avg_completion = (
        user_history["completion_rate"].mean()
        if user_interactions > 0
        else 0.0
    )

    user_active_days = (
        user_history["timestamp"]
        .dt.date
        .nunique()
    )

    user_unique_content = (
        user_history["content_id"]
        .nunique()
    )

    user_unique_creators = (
        user_history["creator_id"]
        .nunique()
    )


    # ======================================
    # STEP 3: GLOBAL PRIOR
    # ======================================

    global_rate = (
        historical_interactions[
            "meaningful_engagement"
        ].mean()
    )

    alpha = 10


    # ======================================
    # STEP 4: USER SMOOTHED RATE
    # ======================================

    user_prior_smoothed_engagement_rate = (
        (
            user_engagements
            +
            alpha * global_rate
        )
        /
        (
            user_interactions
            +
            alpha
        )
    )


    # ======================================
    # STEP 5: USER-GENRE STATISTICS
    # ======================================

    user_genre_stats = (
        user_history
        .groupby("genre")
        .agg(
            user_genre_prior_interactions=(
                "content_id",
                "size"
            ),
            user_genre_prior_clicks=(
                "clicked",
                "sum"
            ),
            user_genre_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
    )

    user_genre_stats[
        "user_genre_prior_engagement_rate"
    ] = (
        user_genre_stats[
            "user_genre_prior_engagements"
        ]
        /
        user_genre_stats[
            "user_genre_prior_interactions"
        ]
    )

    user_genre_stats[
        "user_genre_prior_smoothed_engagement_rate"
    ] = (
        (
            user_genre_stats[
                "user_genre_prior_engagements"
            ]
            +
            alpha * global_rate
        )
        /
        (
            user_genre_stats[
                "user_genre_prior_interactions"
            ]
            +
            alpha
        )
    )


    # ======================================
    # STEP 6: CREATOR STATISTICS
    # ======================================

    creator_stats = (
        historical_interactions
        .groupby("creator_id")
        .agg(
            creator_prior_interactions=(
                "content_id",
                "size"
            ),
            creator_prior_clicks=(
                "clicked",
                "sum"
            ),
            creator_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
    )

    creator_stats[
        "creator_prior_ctr"
    ] = (
        creator_stats["creator_prior_clicks"]
        /
        creator_stats["creator_prior_interactions"]
    )

    creator_stats[
        "creator_prior_engagement_rate"
    ] = (
        creator_stats["creator_prior_engagements"]
        /
        creator_stats["creator_prior_interactions"]
    )

    creator_stats[
        "creator_prior_smoothed_engagement_rate"
    ] = (
        (
            creator_stats[
                "creator_prior_engagements"
            ]
            +
            alpha * global_rate
        )
        /
        (
            creator_stats[
                "creator_prior_interactions"
            ]
            +
            alpha
        )
    )


    # ======================================
    # STEP 7: CONTENT STATISTICS
    # ======================================

    content_stats = (
        historical_interactions
        .groupby("content_id")
        .agg(
            content_prior_interactions=(
                "user_id",
                "size"
            ),
            content_prior_clicks=(
                "clicked",
                "sum"
            ),
            content_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
    )

    content_stats[
        "content_prior_ctr"
    ] = (
        content_stats["content_prior_clicks"]
        /
        content_stats["content_prior_interactions"]
    )

    content_stats[
        "content_prior_engagement_rate"
    ] = (
        content_stats["content_prior_engagements"]
        /
        content_stats["content_prior_interactions"]
    )

    content_stats[
        "content_prior_smoothed_engagement_rate"
    ] = (
        (
            content_stats[
                "content_prior_engagements"
            ]
            +
            alpha * global_rate
        )
        /
        (
            content_stats[
                "content_prior_interactions"
            ]
            +
            alpha
        )
    )


    # ======================================
    # STEP 8: BUILD CANDIDATE FEATURES
    # ======================================

    candidate_features = candidates[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration"
        ]
    ].copy()


    # --------------------------------------
    # Content creation date
    # --------------------------------------

    candidate_features = candidate_features.merge(
        content[
            [
                "content_id",
                "created_at"
            ]
        ],
        on="content_id",
        how="left"
    )


    # --------------------------------------
    # Creator static information
    # --------------------------------------

    candidate_features = candidate_features.merge(
        creators[
            [
                "creator_id",
                "followers",
                "creator_type"
            ]
        ],
        on="creator_id",
        how="left"
    )

    candidate_features = candidate_features.rename(
        columns={
            "followers": "creator_followers"
        }
    )


    # --------------------------------------
    # User static information
    # --------------------------------------

    candidate_features["age_group"] = (
        user_row["age_group"]
    )

    candidate_features["country"] = (
        user_row["country"]
    )

    candidate_features["following_count"] = (
        user_row["following_count"]
    )

    candidate_features["creator_flag"] = (
        user_row["creator_flag"]
    )


    # ======================================
    # STEP 9: CONTEXT FEATURES
    # ======================================

    candidate_features[
        "content_age_days"
    ] = (
        cutoff
        -
        candidate_features["created_at"]
    ).dt.total_seconds() / (
        24 * 60 * 60
    )

    candidate_features["freshness"] = np.exp(
        -candidate_features["content_age_days"] / 30
    )

    candidate_features[
        "interaction_hour"
    ] = cutoff.hour

    candidate_features[
        "interaction_day_of_week"
    ] = cutoff.dayofweek

    candidate_features[
        "is_weekend"
    ] = int(
        cutoff.dayofweek >= 5
    )

    candidate_features[
        "hour_sin"
    ] = np.sin(
        2 * np.pi * cutoff.hour / 24
    )

    candidate_features[
        "hour_cos"
    ] = np.cos(
        2 * np.pi * cutoff.hour / 24
    )

    candidate_features[
        "user_tenure_days"
    ] = (
        cutoff
        -
        pd.Timestamp(
            user_row["signup_date"]
        )
    ).days


    # ======================================
    # STEP 10: USER HISTORY FEATURES
    # ======================================

    candidate_features[
        "user_prior_interactions"
    ] = user_interactions

    candidate_features[
        "user_prior_clicks"
    ] = user_clicks

    candidate_features[
        "user_prior_engagements"
    ] = user_engagements

    candidate_features[
        "user_prior_watch_time"
    ] = user_watch_time

    candidate_features[
        "user_prior_completions"
    ] = user_completions

    candidate_features[
        "user_prior_ctr"
    ] = user_ctr

    candidate_features[
        "user_prior_engagement_rate"
    ] = user_engagement_rate

    candidate_features[
        "user_prior_avg_watch_time"
    ] = user_avg_watch

    candidate_features[
        "user_prior_avg_completion"
    ] = user_avg_completion

    candidate_features[
        "user_prior_active_days"
    ] = user_active_days

    candidate_features[
        "user_prior_unique_content"
    ] = user_unique_content

    candidate_features[
        "user_prior_unique_creators"
    ] = user_unique_creators

    candidate_features[
        "user_prior_smoothed_engagement_rate"
    ] = user_prior_smoothed_engagement_rate


    # ======================================
    # STEP 11: CREATOR HISTORY
    # ======================================

    candidate_features = candidate_features.merge(
        creator_stats,
        left_on="creator_id",
        right_index=True,
        how="left"
    )


    # ======================================
    # STEP 12: CONTENT HISTORY
    # ======================================

    candidate_features = candidate_features.merge(
        content_stats,
        left_on="content_id",
        right_index=True,
        how="left"
    )


    # ======================================
    # STEP 13: USER-GENRE HISTORY
    # ======================================

    candidate_features = candidate_features.merge(
        user_genre_stats,
        left_on="genre",
        right_index=True,
        how="left"
    )


    # ======================================
    # STEP 14: USER-CONTENT HISTORY
    # ======================================

    # Candidates have already been filtered
    # to unseen content. Therefore all
    # user-content historical interaction
    # features are zero.

    candidate_features[
        "user_content_prior_interactions"
    ] = 0

    candidate_features[
        "user_content_prior_clicks"
    ] = 0

    candidate_features[
        "user_content_prior_engagements"
    ] = 0

    candidate_features[
        "user_content_prior_ctr"
    ] = 0.0

    candidate_features[
        "user_content_prior_engagement_rate"
    ] = 0.0

    candidate_features[
        "user_content_prior_smoothed_engagement_rate"
    ] = global_rate


    # ======================================
    # STEP 15: HANDLE COLD-START CREATOR /
    # CONTENT / GENRE FEATURES
    # ======================================

    rate_columns = [
        "creator_prior_ctr",
        "creator_prior_engagement_rate",
        "creator_prior_smoothed_engagement_rate",
        "content_prior_ctr",
        "content_prior_engagement_rate",
        "content_prior_smoothed_engagement_rate",
        "user_genre_prior_engagement_rate",
        "user_genre_prior_smoothed_engagement_rate"
    ]

    count_columns = [
        "creator_prior_interactions",
        "creator_prior_clicks",
        "creator_prior_engagements",
        "content_prior_interactions",
        "content_prior_clicks",
        "content_prior_engagements",
        "user_genre_prior_interactions",
        "user_genre_prior_clicks",
        "user_genre_prior_engagements"
    ]

    for col in rate_columns:
        candidate_features[col] = (
            candidate_features[col]
            .fillna(global_rate)
        )

    for col in count_columns:
        candidate_features[col] = (
            candidate_features[col]
            .fillna(0)
        )


    # ======================================
    # STEP 16: MODEL FEATURE CONTRACT
    # ======================================

    missing_features = [
        col
        for col in feature_columns
        if col not in candidate_features.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing model features: {missing_features}"
        )

    model_features = candidate_features[
        feature_columns
    ].copy()


    # ======================================
    # STEP 17: FINAL VALIDATION
    # ======================================

    if model_features.shape[1] != len(
        feature_columns
    ):
        raise ValueError(
            "Model feature count mismatch."
        )

    if model_features.isnull().any().any():
        missing_values = (
            model_features
            .columns[
                model_features.isnull().any()
            ]
            .tolist()
        )

        raise ValueError(
            f"Missing values in model features: "
            f"{missing_values}"
        )


    # ======================================
    # STEP 18: ENGAGEMENT PREDICTION
    # ======================================

    candidate_features[
        "engagement_score"
    ] = random_forest.predict_proba(
        model_features
    )[:, 1]


    # ======================================
    # STEP 19: RANK CANDIDATES
    # ======================================

    ranked = (
        candidate_features
        .sort_values(
            "engagement_score",
            ascending=False
        )
        .head(k)
    )

    return ranked[
        "content_id"
    ].tolist()


print(
    "✓ Personalized ML recommender function created"
)

✓ Personalized ML recommender function created


In [253]:
# ==========================================
# TEST PERSONALIZED ML RECOMMENDER
# ==========================================

ml_recs = recommend_ml(
    test_user,
    k=K
)

print("Test user:", test_user)

print("\nML recommendations:")
print(ml_recs)

print("\nNumber of recommendations:")
print(len(ml_recs))

print("\nSeen-content overlap:")
print(
    len(
        set(ml_recs)
        &
        seen_content_by_user.get(
            test_user,
            set()
        )
    )
)

print("\n✓ Personalized ML recommender tested")

Test user: U00001

ML recommendations:
['CT001488', 'CT001523', 'CT006944', 'CT006338', 'CT004866', 'CT009762', 'CT006446', 'CT002709', 'CT008295', 'CT004832']

Number of recommendations:
10

Seen-content overlap:
0

✓ Personalized ML recommender tested


In [254]:
# ==========================================
# EVALUATE PERSONALIZED ML RECOMMENDER
# ==========================================

print("Evaluating personalized ML recommender...")
print("This may take some time because each user")
print("requires candidate generation + feature construction + ML scoring.")

ml_results = evaluate_recommender(
    recommend_ml,
    evaluation_users,
    ground_truth,
    K
)

ml_results_df = pd.DataFrame([
    {
        "Strategy": "Personalized ML",
        **ml_results
    }
])

print("\n==========================================")
print("PERSONALIZED ML RESULTS")
print("==========================================")

print(
    ml_results_df.to_string(
        index=False
    )
)

Evaluating personalized ML recommender...
This may take some time because each user
requires candidate generation + feature construction + ML scoring.

PERSONALIZED ML RESULTS
       Strategy  Precision@10  Recall@10  MAP@10  NDCG@10  Users evaluated
Personalized ML        0.0014     0.0055  0.0019   0.0035             3642


In [255]:
# ==========================================
# CANDIDATE RETRIEVAL RECALL@200
# ==========================================

CANDIDATE_K = 200

candidate_recall_hits = 0
candidate_recall_total = 0

candidate_recall_values = []

print("Evaluating candidate retrieval recall...")
print(f"Candidate pool size: {CANDIDATE_K}")
print(f"Evaluation users: {len(evaluation_users)}")


for i, user_id in enumerate(evaluation_users, start=1):

    # --------------------------------------
    # Future relevant items for this user
    # --------------------------------------

    relevant_items = ground_truth.get(
        user_id,
        set()
    )

    if len(relevant_items) == 0:
        continue


    # --------------------------------------
    # Generate content-based candidates
    # --------------------------------------

    candidate_ids = recommend_content_based(
        user_id,
        CANDIDATE_K
    )

    candidate_set = set(candidate_ids)


    # --------------------------------------
    # Calculate candidate recall
    # --------------------------------------

    hits = len(
        relevant_items.intersection(
            candidate_set
        )
    )

    recall = (
        hits / len(relevant_items)
    )

    candidate_recall_values.append(
        recall
    )

    candidate_recall_hits += hits
    candidate_recall_total += len(
        relevant_items
    )


    # Progress indicator
    if i % 500 == 0:
        print(
            f"Processed {i}/{len(evaluation_users)} users..."
        )


# ==========================================
# FINAL METRICS
# ==========================================

macro_candidate_recall = (
    np.mean(candidate_recall_values)
    if candidate_recall_values
    else 0.0
)

micro_candidate_recall = (
    candidate_recall_hits
    /
    candidate_recall_total
    if candidate_recall_total > 0
    else 0.0
)

print("\n==========================================")
print("CANDIDATE RETRIEVAL RESULTS")
print("==========================================")

print(
    f"Macro Candidate Recall@{CANDIDATE_K}: "
    f"{macro_candidate_recall:.4f}"
)

print(
    f"Micro Candidate Recall@{CANDIDATE_K}: "
    f"{micro_candidate_recall:.4f}"
)

print(
    f"Relevant items evaluated: "
    f"{candidate_recall_total:,}"
)

print(
    f"Relevant items retrieved: "
    f"{candidate_recall_hits:,}"
)

Evaluating candidate retrieval recall...
Candidate pool size: 200
Evaluation users: 3642
Processed 500/3642 users...
Processed 1000/3642 users...
Processed 1500/3642 users...
Processed 2000/3642 users...
Processed 2500/3642 users...
Processed 3000/3642 users...
Processed 3500/3642 users...

CANDIDATE RETRIEVAL RESULTS
Macro Candidate Recall@200: 0.0263
Micro Candidate Recall@200: 0.0261
Relevant items evaluated: 10,286
Relevant items retrieved: 268


## Candidate Retrieval Diagnostic

The initial content-based retrieval stage achieved:

- Macro Recall@200: 2.63%
- Micro Recall@200: 2.61%
- Relevant future items: 10,286
- Retrieved relevant items: 268

This indicates that candidate generation is the primary bottleneck
in the current recommendation pipeline.

The personalized ML ranker can only rank items that are present
in the candidate pool. Therefore, improving ranking alone cannot
recover relevant items that were missed during retrieval.

To address this limitation, the candidate-generation stage is
upgraded from a single content-based retriever to a hybrid retrieval
architecture combining content similarity, recent popularity,
global popularity, and exploration candidates.

This follows a standard retrieval → ranking → re-ranking design.

In [256]:
# ==========================================
# HYBRID CANDIDATE GENERATION
# ==========================================

HYBRID_CONTENT_K = 200
HYBRID_RECENT_K = 200
HYBRID_GLOBAL_K = 100


def recommend_hybrid_candidates(
    user_id,
    content_k=HYBRID_CONTENT_K,
    recent_k=HYBRID_RECENT_K,
    global_k=HYBRID_GLOBAL_K
):
    """
    Generate a hybrid candidate pool using:

        1. Content-based personalization
        2. Recent popularity
        3. Global popularity

    Seen content is removed before returning candidates.
    """

    # ======================================
    # STEP 1: USER SEEN CONTENT
    # ======================================

    seen_items = seen_content_by_user.get(
        user_id,
        set()
    )


    # ======================================
    # STEP 2: CONTENT-BASED CANDIDATES
    # ======================================

    content_candidates = recommend_content_based(
        user_id,
        content_k
    )


    # ======================================
    # STEP 3: RECENT POPULARITY
    # ======================================

    recent_candidates = recommend_recent_popular(
        user_id,
        recent_k
    )


    # ======================================
    # STEP 4: GLOBAL POPULARITY
    # ======================================

    global_candidates = recommend_popular(
        user_id,
        global_k
    )


    # ======================================
    # STEP 5: UNION + DEDUPLICATION
    # ======================================

    hybrid_candidates = []

    seen_candidates = set()

    for candidate_list in [
        content_candidates,
        recent_candidates,
        global_candidates
    ]:

        for content_id in candidate_list:

            if content_id in seen_items:
                continue

            if content_id in seen_candidates:
                continue

            hybrid_candidates.append(
                content_id
            )

            seen_candidates.add(
                content_id
            )


    return hybrid_candidates

In [257]:
# ==========================================
# TEST HYBRID CANDIDATE GENERATION
# ==========================================

test_user = "U00001"

hybrid_candidates = recommend_hybrid_candidates(
    test_user
)

print(
    f"User: {test_user}"
)

print(
    f"Hybrid candidate count: "
    f"{len(hybrid_candidates)}"
)

print(
    f"First 20 candidates:"
)

print(
    hybrid_candidates[:20]
)

print(
    f"Seen-content overlap: "
    f"{len(set(hybrid_candidates) & seen_content_by_user.get(test_user, set()))}"
)

User: U00001
Hybrid candidate count: 212
First 20 candidates:
['CT005145', 'CT004464', 'CT001970', 'CT001872', 'CT009442', 'CT003423', 'CT008523', 'CT002147', 'CT003864', 'CT009963', 'CT001165', 'CT004453', 'CT004376', 'CT001446', 'CT000094', 'CT002735', 'CT007697', 'CT006596', 'CT009714', 'CT005616']
Seen-content overlap: 0


In [258]:
# ==========================================
# HYBRID CANDIDATE RETRIEVAL RECALL
# ==========================================

candidate_recall_hits = 0
candidate_recall_total = 0

hybrid_recall_values = []

print("Evaluating hybrid candidate retrieval...")
print(f"Evaluation users: {len(evaluation_users)}")


for i, user_id in enumerate(
    evaluation_users,
    start=1
):

    # --------------------------------------
    # Future relevant items
    # --------------------------------------

    relevant_items = ground_truth.get(
        user_id,
        set()
    )

    if len(relevant_items) == 0:
        continue


    # --------------------------------------
    # Hybrid candidates
    # --------------------------------------

    candidate_ids = recommend_hybrid_candidates(
        user_id
    )

    candidate_set = set(
        candidate_ids
    )


    # --------------------------------------
    # Hits
    # --------------------------------------

    hits = len(
        relevant_items.intersection(
            candidate_set
        )
    )

    recall = (
        hits / len(relevant_items)
    )

    hybrid_recall_values.append(
        recall
    )

    candidate_recall_hits += hits

    candidate_recall_total += len(
        relevant_items
    )


    # --------------------------------------
    # Progress
    # --------------------------------------

    if i % 500 == 0:

        print(
            f"Processed "
            f"{i}/{len(evaluation_users)} users..."
        )


# ==========================================
# FINAL RESULTS
# ==========================================

macro_hybrid_recall = (
    np.mean(hybrid_recall_values)
    if hybrid_recall_values
    else 0.0
)

micro_hybrid_recall = (
    candidate_recall_hits
    /
    candidate_recall_total
    if candidate_recall_total > 0
    else 0.0
)


print("\n==========================================")
print("HYBRID CANDIDATE RETRIEVAL RESULTS")
print("==========================================")

print(
    f"Macro Candidate Recall: "
    f"{macro_hybrid_recall:.4f}"
)

print(
    f"Micro Candidate Recall: "
    f"{micro_hybrid_recall:.4f}"
)

print(
    f"Relevant items evaluated: "
    f"{candidate_recall_total:,}"
)

print(
    f"Relevant items retrieved: "
    f"{candidate_recall_hits:,}"
)

Evaluating hybrid candidate retrieval...
Evaluation users: 3642
Processed 500/3642 users...
Processed 1000/3642 users...
Processed 1500/3642 users...
Processed 2000/3642 users...
Processed 2500/3642 users...
Processed 3000/3642 users...
Processed 3500/3642 users...

HYBRID CANDIDATE RETRIEVAL RESULTS
Macro Candidate Recall: 0.0263
Micro Candidate Recall: 0.0261
Relevant items evaluated: 10,286
Relevant items retrieved: 268


In [259]:
# ==========================================
# RETRIEVAL STRATEGY OVERLAP DIAGNOSTIC
# ==========================================

test_user = "U00001"

content_candidates = set(
    recommend_content_based(
        test_user,
        HYBRID_CONTENT_K
    )
)

recent_candidates = set(
    recommend_recent_popular(
        test_user,
        HYBRID_RECENT_K
    )
)

global_candidates = set(
    recommend_popular(
        test_user,
        HYBRID_GLOBAL_K
    )
)

print("==========================================")
print("RETRIEVAL OVERLAP")
print("==========================================")

print(
    f"Content-Based candidates: "
    f"{len(content_candidates)}"
)

print(
    f"Recent Popularity candidates: "
    f"{len(recent_candidates)}"
)

print(
    f"Global Popularity candidates: "
    f"{len(global_candidates)}"
)

print("\nPairwise overlap:")

print(
    f"Content ∩ Recent: "
    f"{len(content_candidates & recent_candidates)}"
)

print(
    f"Content ∩ Global: "
    f"{len(content_candidates & global_candidates)}"
)

print(
    f"Recent ∩ Global: "
    f"{len(recent_candidates & global_candidates)}"
)

print("\nUnique contribution:")

print(
    f"Recent-only candidates: "
    f"{len(recent_candidates - content_candidates)}"
)

print(
    f"Global-only candidates: "
    f"{len(global_candidates - content_candidates - recent_candidates)}"
)

print("\nTotal union:")

all_candidates = (
    content_candidates
    | recent_candidates
    | global_candidates
)

print(
    f"Unique candidates: "
    f"{len(all_candidates)}"
)

RETRIEVAL OVERLAP
Content-Based candidates: 200
Recent Popularity candidates: 12
Global Popularity candidates: 12

Pairwise overlap:
Content ∩ Recent: 0
Content ∩ Global: 0
Recent ∩ Global: 12

Unique contribution:
Recent-only candidates: 12
Global-only candidates: 0

Total union:
Unique candidates: 212


In [260]:
# ==========================================
# POPULARITY CANDIDATE DIAGNOSTIC
# ==========================================

print("==========================================")
print("POPULARITY CANDIDATE DIAGNOSTIC")
print("==========================================")

print(
    f"Global popularity table rows: "
    f"{len(global_popularity)}"
)

print(
    f"Recent popularity table rows: "
    f"{len(recent_popularity)}"
)

print(
    f"Requested global K: {HYBRID_GLOBAL_K}"
)

print(
    f"Requested recent K: {HYBRID_RECENT_K}"
)

print("\nFor test user:")

print(
    f"Global recommendations: "
    f"{len(recommend_popular(test_user, HYBRID_GLOBAL_K))}"
)

print(
    f"Recent recommendations: "
    f"{len(recommend_recent_popular(test_user, HYBRID_RECENT_K))}"
)

print("\nGlobal top 20 content IDs:")
print(
    global_popularity[
        "content_id"
    ].head(20).tolist()
)

print("\nRecent top 20 content IDs:")
print(
    recent_popularity[
        "content_id"
    ].head(20).tolist()
)

POPULARITY CANDIDATE DIAGNOSTIC
Global popularity table rows: 9638
Recent popularity table rows: 8907
Requested global K: 100
Requested recent K: 200

For test user:
Global recommendations: 100
Recent recommendations: 200

Global top 20 content IDs:
['CT001010', 'CT003051', 'CT007858', 'CT009767', 'CT006791', 'CT004242', 'CT008086', 'CT001063', 'CT002446', 'CT005668', 'CT008770', 'CT005531', 'CT006487', 'CT006580', 'CT005594', 'CT009361', 'CT000834', 'CT002178', 'CT003797', 'CT002297']

Recent top 20 content IDs:
['CT009210', 'CT002215', 'CT001900', 'CT002868', 'CT002295', 'CT005537', 'CT008155', 'CT007293', 'CT001930', 'CT007014', 'CT006963', 'CT009794', 'CT000756', 'CT007272', 'CT004938', 'CT003404', 'CT006352', 'CT005973', 'CT005102', 'CT002335']


In [261]:
# ==========================================
# RECOMMENDER COVERAGE / DIVERSITY / NOVELTY
# ==========================================

def evaluate_catalog_metrics(
    recommendation_function,
    users_to_evaluate,
    k=10
):
    """
    Evaluate catalog coverage, creator diversity,
    genre diversity and recommendation novelty.
    """

    recommended_items = []
    creator_diversities = []
    genre_diversities = []
    novelty_scores = []

    # --------------------------------------
    # Popularity lookup for novelty
    # --------------------------------------

    popularity_lookup = (
        global_popularity
        .set_index("content_id")[
            "interaction_count"
        ]
        .to_dict()
    )

    total_interactions = (
        historical_interactions.shape[0]
    )

    # --------------------------------------
    # Evaluate users
    # --------------------------------------

    for user_id in users_to_evaluate:

        recs = recommendation_function(
            user_id,
            k
        )

        if len(recs) == 0:
            continue

        recs = recs[:k]

        recommended_items.extend(recs)

        # ----------------------------------
        # Metadata
        # ----------------------------------

        rec_metadata = content[
            content["content_id"].isin(recs)
        ][
            [
                "content_id",
                "creator_id",
                "genre"
            ]
        ]

        # ----------------------------------
        # Creator diversity
        # ----------------------------------

        creator_diversity = (
            rec_metadata["creator_id"]
            .nunique()
            /
            len(recs)
        )

        creator_diversities.append(
            creator_diversity
        )

        # ----------------------------------
        # Genre diversity
        # ----------------------------------

        genre_diversity = (
            rec_metadata["genre"]
            .nunique()
            /
            len(recs)
        )

        genre_diversities.append(
            genre_diversity
        )

        # ----------------------------------
        # Novelty
        #
        # Less popular item = more novel
        # ----------------------------------

        item_novelty = []

        for content_id in recs:

            count = popularity_lookup.get(
                content_id,
                0
            )

            # Smoothed popularity
            popularity_probability = (
                (count + 1)
                /
                (
                    total_interactions
                    +
                    len(popularity_lookup)
                )
            )

            novelty = -np.log2(
                popularity_probability
            )

            item_novelty.append(
                novelty
            )

        novelty_scores.append(
            np.mean(item_novelty)
        )

    # --------------------------------------
    # Catalog coverage
    # --------------------------------------

    unique_recommended_items = len(
        set(recommended_items)
    )

    eligible_catalog_size = len(
        eligible_content
    )

    catalog_coverage = (
        unique_recommended_items
        /
        eligible_catalog_size
    )

    return {
        "Catalog Coverage": catalog_coverage,
        "Unique Recommended Items": unique_recommended_items,
        "Creator Diversity": np.mean(
            creator_diversities
        ),
        "Genre Diversity": np.mean(
            genre_diversities
        ),
        "Novelty": np.mean(
            novelty_scores
        )
    }

In [262]:
# ==========================================
# COMPARE CATALOG METRICS
# ==========================================

print("Evaluating catalog metrics...")
print("This may take some time.")


catalog_results = []


strategies = {
    "Global Popularity": recommend_popular,
    "Recent Popularity": recommend_recent_popular,
    "Content-Based": recommend_content_based,
    "Personalized ML": recommend_ml
}


for strategy_name, recommendation_function in strategies.items():

    print(
        f"\nEvaluating: {strategy_name}"
    )

    metrics = evaluate_catalog_metrics(
        recommendation_function,
        evaluation_users,
        k=10
    )

    metrics["Strategy"] = strategy_name

    catalog_results.append(
        metrics
    )


catalog_results_df = pd.DataFrame(
    catalog_results
)

catalog_results_df = catalog_results_df[
    [
        "Strategy",
        "Catalog Coverage",
        "Unique Recommended Items",
        "Creator Diversity",
        "Genre Diversity",
        "Novelty"
    ]
]


print("\n==========================================")
print("CATALOG / DIVERSITY RESULTS")
print("==========================================")

print(
    catalog_results_df.to_string(
        index=False
    )
)

Evaluating catalog metrics...
This may take some time.

Evaluating: Global Popularity

Evaluating: Recent Popularity

Evaluating: Content-Based

Evaluating: Personalized ML

CATALOG / DIVERSITY RESULTS
         Strategy  Catalog Coverage  Unique Recommended Items  Creator Diversity  Genre Diversity  Novelty
Global Popularity            0.0012                        12             0.0000           0.0000  17.7611
Recent Popularity            0.0012                        12             0.0000           0.0000  17.7611
    Content-Based            0.8836                      8535             0.5134           0.3300  13.1934
  Personalized ML            0.5234                      5056             0.8508           0.2434  13.8911


In [269]:
# ==========================================
# COLD-START AWARE RECOMMENDER
# ==========================================

def recommend_ml(
    user_id,
    k=10,
    candidate_k=ML_CANDIDATE_K
):
    """
    Cold-start-aware personalized recommender.

    Existing users:
        Content-based candidates
        -> engagement model
        -> personalized ranking

    New users:
        Recent popularity
        -> global popularity fallback

    Returns:
        List of content IDs.
    """

    # ======================================
    # STEP 1: CHECK USER HISTORY
    # ======================================

    user_history = historical_interactions[
        historical_interactions["user_id"] == user_id
    ].copy()

    # ======================================
    # STEP 1A: COLD-START USER
    # ======================================

    if len(user_history) == 0:

        # ----------------------------------
        # Recent popularity fallback
        # ----------------------------------

        recent_result = recommend_recent_popular(
            user_id,
            max(k * 3, 30)
        )

        # Make sure we always extract content IDs
        if isinstance(recent_result, pd.DataFrame):

            fallback_candidates = (
                recent_result["content_id"]
                .tolist()
            )

        else:

            fallback_candidates = list(
                recent_result
            )

        # ----------------------------------
        # Global popularity fallback
        # ----------------------------------

        if len(fallback_candidates) < k:

            global_result = recommend_popular(
                user_id,
                max(k * 3, 30)
            )

            if isinstance(global_result, pd.DataFrame):

                global_candidates = (
                    global_result["content_id"]
                    .tolist()
                )

            else:

                global_candidates = list(
                    global_result
                )

            existing = set(
                fallback_candidates
            )

            for content_id in global_candidates:

                if content_id not in existing:

                    fallback_candidates.append(
                        content_id
                    )

                    existing.add(
                        content_id
                    )

                if len(fallback_candidates) >= k:
                    break

        return fallback_candidates[:k]

    # ======================================
    # STEP 2: PERSONALIZED CANDIDATES
    # ======================================

    candidate_ids = recommend_content_based(
        user_id,
        candidate_k
    )

    # Handle either DataFrame or list output
    if isinstance(candidate_ids, pd.DataFrame):

        candidate_ids = (
            candidate_ids["content_id"]
            .tolist()
        )

    else:

        candidate_ids = list(
            candidate_ids
        )

    if len(candidate_ids) == 0:
        return []

    # ======================================
    # STEP 3: REMOVE SEEN CONTENT
    # ======================================

    seen_content = set(
        user_history["content_id"]
    )

    candidate_ids = [
        content_id
        for content_id in candidate_ids
        if content_id not in seen_content
    ]

    if len(candidate_ids) == 0:
        return []

    # ======================================
    # STEP 4: CANDIDATE METADATA
    # ======================================

    candidate_features = content[
        content["content_id"].isin(candidate_ids)
    ][
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration"
        ]
    ].copy()

    candidate_features = (
        candidate_features
        .drop_duplicates("content_id")
        .reset_index(drop=True)
    )

    # ======================================
    # STEP 5: CREATOR FEATURES
    # ======================================

    creator_columns = [
        "creator_id"
    ]

    if "followers" in creators.columns:
        creator_columns.append("followers")

    elif "creator_followers" in creators.columns:
        creator_columns.append("creator_followers")

    creator_lookup = creators[
        creator_columns
    ].drop_duplicates("creator_id")

    candidate_features = candidate_features.merge(
        creator_lookup,
        on="creator_id",
        how="left"
    )

    if "followers" in candidate_features.columns:

        candidate_features = (
            candidate_features
            .rename(
                columns={
                    "followers":
                    "creator_followers"
                }
            )
        )

    # ======================================
    # STEP 6: CREATOR TYPE
    # ======================================

    candidate_features["creator_type"] = np.where(
        candidate_features[
            "creator_followers"
        ].fillna(0) >= 500,
        "Established Creator",
        "Emerging Creator"
    )

    # ======================================
    # STEP 7: CONTENT AGE / FRESHNESS
    # ======================================

    recommendation_time = (
        historical_interactions["timestamp"].max()
    )

    candidate_features = candidate_features.merge(
        content[
            [
                "content_id",
                "created_at"
            ]
        ],
        on="content_id",
        how="left"
    )

    candidate_features[
        "content_age_days"
    ] = (
        recommendation_time
        - candidate_features["created_at"]
    ).dt.total_seconds() / (
        24 * 60 * 60
    )

    candidate_features[
        "content_age_days"
    ] = (
        candidate_features[
            "content_age_days"
        ].clip(lower=0)
    )

    candidate_features[
        "freshness"
    ] = np.exp(
        -candidate_features[
            "content_age_days"
        ] / 30
    )

    # ======================================
    # STEP 8: CONTEXT FEATURES
    # ======================================

    recommendation_hour = (
        recommendation_time.hour
    )

    recommendation_day = (
        recommendation_time.dayofweek
    )

    candidate_features[
        "interaction_hour"
    ] = recommendation_hour

    candidate_features[
        "interaction_day_of_week"
    ] = recommendation_day

    candidate_features[
        "is_weekend"
    ] = int(
        recommendation_day >= 5
    )

    candidate_features[
        "hour_sin"
    ] = np.sin(
        2 * np.pi * recommendation_hour / 24
    )

    candidate_features[
        "hour_cos"
    ] = np.cos(
        2 * np.pi * recommendation_hour / 24
    )

    # ======================================
    # STEP 9: USER STATIC FEATURES
    # ======================================

    user_row = users[
        users["user_id"] == user_id
    ]

    if len(user_row) == 0:
        return []

    user_row = user_row.iloc[0]

    candidate_features[
        "age_group"
    ] = user_row["age_group"]

    candidate_features[
        "country"
    ] = user_row["country"]

    candidate_features[
        "following_count"
    ] = user_row["following_count"]

    candidate_features[
        "creator_flag"
    ] = user_row["creator_flag"]

    candidate_features[
        "user_tenure_days"
    ] = (
        recommendation_time
        - user_row["signup_date"]
    ).total_seconds() / (
        24 * 60 * 60
    )

    # ======================================
    # STEP 10: USER HISTORY FEATURES
    # ======================================

    user_prior_interactions = len(
        user_history
    )

    user_prior_clicks = (
        user_history["clicked"].sum()
    )

    user_prior_engagements = (
        user_history[
            "meaningful_engagement"
        ].sum()
    )

    user_prior_watch_time = (
        user_history["watch_time"].sum()
    )

    user_prior_completions = (
        user_history["completion_rate"].sum()
    )

    user_prior_ctr = (
        user_prior_clicks
        /
        max(user_prior_interactions, 1)
    )

    user_prior_engagement_rate = (
        user_prior_engagements
        /
        max(user_prior_interactions, 1)
    )

    user_prior_avg_watch_time = (
        user_history["watch_time"].mean()
    )

    user_prior_avg_completion = (
        user_history["completion_rate"].mean()
    )

    user_prior_active_days = (
        user_history["timestamp"]
        .dt.date
        .nunique()
    )

    user_prior_unique_content = (
        user_history["content_id"]
        .nunique()
    )

    user_prior_unique_creators = (
        user_history["creator_id"]
        .nunique()
    )

    candidate_features[
        "user_prior_interactions"
    ] = user_prior_interactions

    candidate_features[
        "user_prior_clicks"
    ] = user_prior_clicks

    candidate_features[
        "user_prior_engagements"
    ] = user_prior_engagements

    candidate_features[
        "user_prior_watch_time"
    ] = user_prior_watch_time

    candidate_features[
        "user_prior_completions"
    ] = user_prior_completions

    candidate_features[
        "user_prior_ctr"
    ] = user_prior_ctr

    candidate_features[
        "user_prior_engagement_rate"
    ] = user_prior_engagement_rate

    candidate_features[
        "user_prior_avg_watch_time"
    ] = user_prior_avg_watch_time

    candidate_features[
        "user_prior_avg_completion"
    ] = user_prior_avg_completion

    candidate_features[
        "user_prior_active_days"
    ] = user_prior_active_days

    candidate_features[
        "user_prior_unique_content"
    ] = user_prior_unique_content

    candidate_features[
        "user_prior_unique_creators"
    ] = user_prior_unique_creators

    # ======================================
    # STEP 11: CREATOR HISTORY
    # ======================================

    creator_stats = (
        historical_interactions
        .groupby("creator_id")
        .agg(
            creator_prior_interactions=(
                "interaction_id",
                "count"
            ),
            creator_prior_clicks=(
                "clicked",
                "sum"
            ),
            creator_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
        .reset_index()
    )

    creator_stats[
        "creator_prior_ctr"
    ] = (
        creator_stats[
            "creator_prior_clicks"
        ]
        /
        creator_stats[
            "creator_prior_interactions"
        ].replace(0, np.nan)
    )

    creator_stats[
        "creator_prior_engagement_rate"
    ] = (
        creator_stats[
            "creator_prior_engagements"
        ]
        /
        creator_stats[
            "creator_prior_interactions"
        ].replace(0, np.nan)
    )

    candidate_features = candidate_features.merge(
        creator_stats,
        on="creator_id",
        how="left"
    )

    # ======================================
    # STEP 12: CONTENT HISTORY
    # ======================================

    content_stats = (
        historical_interactions
        .groupby("content_id")
        .agg(
            content_prior_interactions=(
                "interaction_id",
                "count"
            ),
            content_prior_clicks=(
                "clicked",
                "sum"
            ),
            content_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
        .reset_index()
    )

    content_stats[
        "content_prior_ctr"
    ] = (
        content_stats[
            "content_prior_clicks"
        ]
        /
        content_stats[
            "content_prior_interactions"
        ].replace(0, np.nan)
    )

    content_stats[
        "content_prior_engagement_rate"
    ] = (
        content_stats[
            "content_prior_engagements"
        ]
        /
        content_stats[
            "content_prior_interactions"
        ].replace(0, np.nan)
    )

    candidate_features = candidate_features.merge(
        content_stats,
        on="content_id",
        how="left"
    )

    # ======================================
    # STEP 13: USER-GENRE HISTORY
    # ======================================

    user_genre_stats = (
        historical_interactions
        .groupby(
            ["user_id", "genre"]
        )
        .agg(
            user_genre_prior_interactions=(
                "interaction_id",
                "count"
            ),
            user_genre_prior_clicks=(
                "clicked",
                "sum"
            ),
            user_genre_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
        .reset_index()
    )

    user_genre_stats[
        "user_genre_prior_engagement_rate"
    ] = (
        user_genre_stats[
            "user_genre_prior_engagements"
        ]
        /
        user_genre_stats[
            "user_genre_prior_interactions"
        ].replace(0, np.nan)
    )

    # Only keep this user's genre history
    user_genre_stats = user_genre_stats[
        user_genre_stats["user_id"] == user_id
    ].drop(
        columns=["user_id"]
    )

    candidate_features = candidate_features.merge(
        user_genre_stats,
        on="genre",
        how="left"
    )

    # ======================================
    # STEP 14: USER-CONTENT HISTORY
    # ======================================

    user_content_stats = (
        user_history
        .groupby("content_id")
        .agg(
            user_content_prior_interactions=(
                "interaction_id",
                "count"
            ),
            user_content_prior_clicks=(
                "clicked",
                "sum"
            ),
            user_content_prior_engagements=(
                "meaningful_engagement",
                "sum"
            )
        )
        .reset_index()
    )

    user_content_stats[
        "user_content_prior_ctr"
    ] = (
        user_content_stats[
            "user_content_prior_clicks"
        ]
        /
        user_content_stats[
            "user_content_prior_interactions"
        ].replace(0, np.nan)
    )

    user_content_stats[
        "user_content_prior_engagement_rate"
    ] = (
        user_content_stats[
            "user_content_prior_engagements"
        ]
        /
        user_content_stats[
            "user_content_prior_interactions"
        ].replace(0, np.nan)
    )

    candidate_features = candidate_features.merge(
        user_content_stats,
        on="content_id",
        how="left"
    )

    # ======================================
    # STEP 15: FILL MISSING HISTORY
    # ======================================

    global_rate = (
        historical_interactions[
            "meaningful_engagement"
        ].mean()
    )

    zero_features = [
        "creator_prior_interactions",
        "creator_prior_clicks",
        "creator_prior_engagements",
        "content_prior_interactions",
        "content_prior_clicks",
        "content_prior_engagements",
        "user_content_prior_interactions",
        "user_content_prior_clicks",
        "user_content_prior_engagements",
        "user_genre_prior_interactions",
        "user_genre_prior_clicks",
        "user_genre_prior_engagements"
    ]

    for column in zero_features:

        if column in candidate_features.columns:

            candidate_features[column] = (
                candidate_features[column]
                .fillna(0)
            )

    rate_features = [
        "creator_prior_ctr",
        "creator_prior_engagement_rate",
        "content_prior_ctr",
        "content_prior_engagement_rate",
        "user_content_prior_ctr",
        "user_content_prior_engagement_rate",
        "user_genre_prior_engagement_rate"
    ]

    for column in rate_features:

        if column in candidate_features.columns:

            candidate_features[column] = (
                candidate_features[column]
                .fillna(global_rate)
            )

    # ======================================
    # STEP 16: SMOOTHED RATES
    # ======================================

    alpha = 10

    candidate_features[
        "user_prior_smoothed_engagement_rate"
    ] = (
        user_prior_engagements
        + alpha * global_rate
    ) / (
        user_prior_interactions
        + alpha
    )

    candidate_features[
        "creator_prior_smoothed_engagement_rate"
    ] = (
        candidate_features[
            "creator_prior_engagements"
        ]
        + alpha * global_rate
    ) / (
        candidate_features[
            "creator_prior_interactions"
        ]
        + alpha
    )

    candidate_features[
        "content_prior_smoothed_engagement_rate"
    ] = (
        candidate_features[
            "content_prior_engagements"
        ]
        + alpha * global_rate
    ) / (
        candidate_features[
            "content_prior_interactions"
        ]
        + alpha
    )

    candidate_features[
        "user_content_prior_smoothed_engagement_rate"
    ] = (
        candidate_features[
            "user_content_prior_engagements"
        ]
        + alpha * global_rate
    ) / (
        candidate_features[
            "user_content_prior_interactions"
        ]
        + alpha
    )

    candidate_features[
        "user_genre_prior_smoothed_engagement_rate"
    ] = (
        candidate_features[
            "user_genre_prior_engagements"
        ]
        + alpha * global_rate
    ) / (
        candidate_features[
            "user_genre_prior_interactions"
        ]
        + alpha
    )

    # ======================================
    # STEP 17: MODEL FEATURE CONTRACT
    # ======================================

    feature_columns = engagement_feature_columns

    missing_features = [
        column
        for column in feature_columns
        if column not in candidate_features.columns
    ]

    if missing_features:

        raise ValueError(
            f"Missing model features: "
            f"{missing_features}"
        )

    model_features = (
        candidate_features[
            feature_columns
        ]
        .copy()
    )

    # ======================================
    # STEP 18: MODEL SCORING
    # ======================================

    scores = random_forest.predict_proba(
        model_features
    )[:, 1]

    candidate_features[
        "engagement_score"
    ] = scores

    # ======================================
    # STEP 19: FINAL RANKING
    # ======================================

    candidate_features = (
        candidate_features
        .sort_values(
            "engagement_score",
            ascending=False
        )
    )

    return (
        candidate_features[
            "content_id"
        ]
        .head(k)
        .tolist()
    )


print("✓ Cold-start-aware recommend_ml() defined")

✓ Cold-start-aware recommend_ml() defined


In [270]:
# ==========================================
# COLD-START OUTPUT CONTRACT CHECK
# ==========================================

print("==========================================")
print("COLD-START OUTPUT CONTRACT CHECK")
print("==========================================")

cold_user = new_users[0]

recs = recommend_ml(
    cold_user,
    k=10
)

print("User:", cold_user)
print("Type:", type(recs))
print("Number of recommendations:", len(recs))

if isinstance(recs, list):
    print("First recommendation:", recs[0] if recs else None)
    print(
        "All values are strings:",
        all(isinstance(x, str) for x in recs)
    )
    print(
        "Unique recommendations:",
        len(set(recs))
    )

    assert len(recs) == 10
    assert all(isinstance(x, str) for x in recs)
    assert len(set(recs)) == 10

    print("\n✓ Exactly 10 recommendations returned")
    print("✓ Output is a Python list")
    print("✓ Content IDs are strings")
    print("✓ Recommendations are unique")

else:
    print("\n⚠️ Output is not a Python list.")
    print("Returned object:")
    print(recs)

print("==========================================")

COLD-START OUTPUT CONTRACT CHECK
User: U00005
Type: <class 'list'>
Number of recommendations: 10
First recommendation: CT009210
All values are strings: True
Unique recommendations: 10

✓ Exactly 10 recommendations returned
✓ Output is a Python list
✓ Content IDs are strings
✓ Recommendations are unique


In [280]:
# ============================================================
# FINAL COLD-START-AWARE RECOMMENDER
# ============================================================

import numpy as np
import pandas as pd


def recommend_ml(user_id, k=10, random_state=42):
    """
    Cold-start-aware personalized recommender.

    Cases:
    1. New user:
       Recent popularity -> global popularity fallback

    2. Existing user:
       Content-based candidate generation -> ML engagement ranking

    3. New content:
       Content with no historical interactions is still eligible.
       Historical features default to safe prior values.

    4. New creator:
       Creator historical features default to safe prior values.

    Returns:
        list[str]: exactly up to k unique content IDs
    """

    rng = np.random.default_rng(random_state)

    # --------------------------------------------------------
    # 1. BASIC VALIDATION
    # --------------------------------------------------------

    if user_id not in set(users["user_id"]):
        raise ValueError(f"Unknown user_id: {user_id}")

    if k <= 0:
        return []

    # --------------------------------------------------------
    # 2. USER HISTORY
    # --------------------------------------------------------

    user_history = historical_interactions[
        historical_interactions["user_id"] == user_id
    ].copy()

    seen_content = set(user_history["content_id"])

    # --------------------------------------------------------
    # 3. NEW-USER COLD START
    # --------------------------------------------------------

    if len(user_history) == 0:

        # First preference: recent popularity
        recent_recs = recommend_recent_popular(
            user_id,
            k=k
        )

        if isinstance(recent_recs, pd.DataFrame):
            recent_recs = recent_recs["content_id"].tolist()
        else:
            recent_recs = list(recent_recs)

        recent_recs = [
            str(x)
            for x in recent_recs
            if str(x) not in seen_content
        ]

        if len(recent_recs) >= k:
            return list(dict.fromkeys(recent_recs))[:k]

        # Second preference: global popularity
        global_recs = recommend_popular(
            user_id,
            k=k
        )

        if isinstance(global_recs, pd.DataFrame):
            global_recs = global_recs["content_id"].tolist()
        else:
            global_recs = list(global_recs)

        recommendations = recent_recs + [
            str(x)
            for x in global_recs
            if str(x) not in recent_recs
            and str(x) not in seen_content
        ]

        return list(dict.fromkeys(recommendations))[:k]

    # --------------------------------------------------------
    # 4. EXISTING USER: CONTENT-BASED CANDIDATE GENERATION
    # --------------------------------------------------------

    # User profile from historically engaged content
    positive_history = user_history[
        user_history["meaningful_engagement"] == 1
    ].copy()

    profile_content_ids = positive_history["content_id"].unique()

    profile_indices = [
        content_id_to_index[cid]
        for cid in profile_content_ids
        if cid in content_id_to_index.index
    ]

    if len(profile_indices) > 0:

        # Build the user's TF-IDF profile as a 1D vector
        user_profile = np.asarray(
            content_matrix[profile_indices].mean(axis=0)
        ).ravel()

        # Calculate cosine-like similarity using the normalized
        # TF-IDF representation.
        similarity_scores = np.asarray(
            content_matrix @ user_profile
        ).ravel()

        # Safety check: one score per eligible content item
        if len(similarity_scores) != len(content_features):
            raise ValueError(
                f"Similarity score length {len(similarity_scores)} "
                f"does not match content rows {len(content_features)}"
            )

        content_features_scored = content_features.copy()
        content_features_scored["similarity_score"] = similarity_scores

    else:
        # If user has no meaningful engagement history,
        # fall back to popularity-based candidates.
        recent_recs = recommend_recent_popular(
            user_id,
            k=k
        )

        if isinstance(recent_recs, pd.DataFrame):
            recent_recs = recent_recs["content_id"].tolist()
        else:
            recent_recs = list(recent_recs)

        return list(dict.fromkeys(
            str(x) for x in recent_recs
            if str(x) not in seen_content
        ))[:k]

    # --------------------------------------------------------
    # 5. REMOVE ALREADY-SEEN CONTENT
    # --------------------------------------------------------

    candidate_pool = (
        content_features_scored[
            ~content_features_scored["content_id"].isin(seen_content)
        ]
        .sort_values(
            "similarity_score",
            ascending=False
        )
        .head(300)
        .copy()
    )

    # If content-based retrieval somehow produces no candidates,
    # use recent popularity.
    if len(candidate_pool) == 0:

        recent_recs = recommend_recent_popular(
            user_id,
            k=k
        )

        if isinstance(recent_recs, pd.DataFrame):
            recent_recs = recent_recs["content_id"].tolist()
        else:
            recent_recs = list(recent_recs)

        return list(dict.fromkeys(
            str(x) for x in recent_recs
            if str(x) not in seen_content
        ))[:k]

    # --------------------------------------------------------
    # 6. CURRENT CONTEXT
    # --------------------------------------------------------

    reference_time = interactions["timestamp"].max()

    interaction_hour = reference_time.hour
    interaction_day_of_week = reference_time.dayofweek
    is_weekend = int(interaction_day_of_week >= 5)

    # --------------------------------------------------------
    # 7. USER HISTORICAL FEATURES
    # --------------------------------------------------------

    user_prior_interactions = len(user_history)

    user_prior_clicks = user_history["clicked"].sum()
    user_prior_engagements = user_history["meaningful_engagement"].sum()

    user_prior_watch_time = user_history["watch_time"].sum()
    user_prior_completions = user_history["completion_rate"].sum()

    user_prior_ctr = (
        user_prior_clicks / user_prior_interactions
        if user_prior_interactions > 0
        else 0.5
    )

    user_prior_engagement_rate = (
        user_prior_engagements / user_prior_interactions
        if user_prior_interactions > 0
        else 0.5
    )

    user_prior_avg_watch_time = (
        user_history["watch_time"].mean()
        if user_prior_interactions > 0
        else 0.0
    )

    user_prior_avg_completion = (
        user_history["completion_rate"].mean()
        if user_prior_interactions > 0
        else 0.0
    )

    user_prior_active_days = (
        user_history["timestamp"].dt.date.nunique()
    )

    user_prior_unique_content = (
        user_history["content_id"].nunique()
    )

    user_prior_unique_creators = (
        user_history["creator_id"].nunique()
    )

    # --------------------------------------------------------
    # 8. USER STATIC FEATURES
    # --------------------------------------------------------

    user_row = users[
        users["user_id"] == user_id
    ].iloc[0]

    # --------------------------------------------------------
    # 9. CREATOR HISTORICAL FEATURES
    # --------------------------------------------------------

    creator_stats = (
        historical_interactions
        .groupby("creator_id")
        .agg(
            creator_prior_interactions=("user_id", "size"),
            creator_prior_clicks=("clicked", "sum"),
            creator_prior_engagements=("meaningful_engagement", "sum")
        )
        .reset_index()
    )

    creator_stats["creator_prior_ctr"] = (
        creator_stats["creator_prior_clicks"]
        / creator_stats["creator_prior_interactions"]
    )

    creator_stats["creator_prior_engagement_rate"] = (
        creator_stats["creator_prior_engagements"]
        / creator_stats["creator_prior_interactions"]
    )

    # --------------------------------------------------------
    # 10. CONTENT HISTORICAL FEATURES
    # --------------------------------------------------------

    content_stats = (
        historical_interactions
        .groupby("content_id")
        .agg(
            content_prior_interactions=("user_id", "size"),
            content_prior_clicks=("clicked", "sum"),
            content_prior_engagements=("meaningful_engagement", "sum")
        )
        .reset_index()
    )

    content_stats["content_prior_ctr"] = (
        content_stats["content_prior_clicks"]
        / content_stats["content_prior_interactions"]
    )

    content_stats["content_prior_engagement_rate"] = (
        content_stats["content_prior_engagements"]
        / content_stats["content_prior_interactions"]
    )

    # --------------------------------------------------------
    # 11. USER-CONTENT HISTORY
    # --------------------------------------------------------

    user_content_stats = (
        user_history
        .groupby("content_id")
        .agg(
            user_content_prior_interactions=("user_id", "size"),
            user_content_prior_clicks=("clicked", "sum"),
            user_content_prior_engagements=("meaningful_engagement", "sum")
        )
        .reset_index()
    )

    user_content_stats["user_content_prior_ctr"] = (
        user_content_stats["user_content_prior_clicks"]
        / user_content_stats["user_content_prior_interactions"]
    )

    user_content_stats["user_content_prior_engagement_rate"] = (
        user_content_stats["user_content_prior_engagements"]
        / user_content_stats["user_content_prior_interactions"]
    )

    # --------------------------------------------------------
    # 12. USER-GENRE HISTORY
    # --------------------------------------------------------

    user_genre_stats = (
        user_history
        .groupby("genre")
        .agg(
            user_genre_prior_interactions=("user_id", "size"),
            user_genre_prior_clicks=("clicked", "sum"),
            user_genre_prior_engagements=("meaningful_engagement", "sum")
        )
        .reset_index()
    )

    user_genre_stats["user_genre_prior_engagement_rate"] = (
        user_genre_stats["user_genre_prior_engagements"]
        / user_genre_stats["user_genre_prior_interactions"]
    )

    # --------------------------------------------------------
    # 13. GLOBAL PRIOR
    # --------------------------------------------------------

    global_prior_engagement_rate = (
        historical_interactions["meaningful_engagement"].mean()
    )

    alpha = 10

    # --------------------------------------------------------
    # 14. MERGE FEATURES
    # --------------------------------------------------------

    candidate_features = candidate_pool[
        [
            "content_id",
            "creator_id",
            "genre",
            "content_type",
            "duration"
        ]
    ].copy()

    candidate_features = candidate_features.merge(
        eligible_content[
            [
                "content_id",
                "created_at"
            ]
        ],
        on="content_id",
        how="left"
    )

    candidate_features = candidate_features.merge(
        content_stats,
        on="content_id",
        how="left"
    )

    candidate_features = candidate_features.merge(
        creator_stats,
        on="creator_id",
        how="left"
    )

    candidate_features = candidate_features.merge(
        user_content_stats,
        on="content_id",
        how="left"
    )

    candidate_features = candidate_features.merge(
        user_genre_stats,
        on="genre",
        how="left"
    )

    # --------------------------------------------------------
    # 15. COLD-START DEFAULTS
    # --------------------------------------------------------

    count_features = [
        "content_prior_interactions",
        "content_prior_clicks",
        "content_prior_engagements",
        "creator_prior_interactions",
        "creator_prior_clicks",
        "creator_prior_engagements",
        "user_content_prior_interactions",
        "user_content_prior_clicks",
        "user_content_prior_engagements",
        "user_genre_prior_interactions",
        "user_genre_prior_clicks",
        "user_genre_prior_engagements"
    ]

    rate_features = [
        "content_prior_ctr",
        "content_prior_engagement_rate",
        "creator_prior_ctr",
        "creator_prior_engagement_rate",
        "user_content_prior_ctr",
        "user_content_prior_engagement_rate",
        "user_genre_prior_engagement_rate"
    ]

    for col in count_features:
        candidate_features[col] = (
            candidate_features[col]
            .fillna(0)
        )

    for col in rate_features:
        candidate_features[col] = (
            candidate_features[col]
            .fillna(global_prior_engagement_rate)
        )

    # --------------------------------------------------------
    # 16. SMOOTHED HISTORICAL RATES
    # --------------------------------------------------------

    candidate_features[
        "content_prior_smoothed_engagement_rate"
    ] = (
        candidate_features["content_prior_engagements"]
        + alpha * global_prior_engagement_rate
    ) / (
        candidate_features["content_prior_interactions"]
        + alpha
    )

    candidate_features[
        "creator_prior_smoothed_engagement_rate"
    ] = (
        candidate_features["creator_prior_engagements"]
        + alpha * global_prior_engagement_rate
    ) / (
        candidate_features["creator_prior_interactions"]
        + alpha
    )

    candidate_features[
        "user_content_prior_smoothed_engagement_rate"
    ] = (
        candidate_features["user_content_prior_engagements"]
        + alpha * global_prior_engagement_rate
    ) / (
        candidate_features["user_content_prior_interactions"]
        + alpha
    )

    candidate_features[
        "user_genre_prior_smoothed_engagement_rate"
    ] = (
        candidate_features["user_genre_prior_engagements"]
        + alpha * global_prior_engagement_rate
    ) / (
        candidate_features["user_genre_prior_interactions"]
        + alpha
    )

    candidate_features[
        "user_prior_smoothed_engagement_rate"
    ] = (
        user_prior_engagements
        + alpha * global_prior_engagement_rate
    ) / (
        user_prior_interactions
        + alpha
    )

    # --------------------------------------------------------
    # 17. TEMPORAL / CONTEXT FEATURES
    # --------------------------------------------------------

    candidate_features["content_age_days"] = (
        reference_time
        - candidate_features["created_at"]
    ).dt.total_seconds() / (24 * 60 * 60)

    candidate_features["content_age_days"] = (
        candidate_features["content_age_days"]
        .clip(lower=0)
    )

    candidate_features["freshness"] = np.exp(
        -candidate_features["content_age_days"] / 30
    )

    candidate_features["interaction_hour"] = interaction_hour

    candidate_features["interaction_day_of_week"] = (
        interaction_day_of_week
    )

    candidate_features["is_weekend"] = is_weekend

    candidate_features["hour_sin"] = np.sin(
        2 * np.pi * interaction_hour / 24
    )

    candidate_features["hour_cos"] = np.cos(
        2 * np.pi * interaction_hour / 24
    )

    candidate_features["user_tenure_days"] = (
        reference_time
        - pd.Timestamp(user_row["signup_date"])
    ).total_seconds() / (24 * 60 * 60)

    # --------------------------------------------------------
    # 18. USER FEATURES
    # --------------------------------------------------------

    candidate_features["age_group"] = user_row["age_group"]
    candidate_features["country"] = user_row["country"]
    candidate_features["following_count"] = user_row["following_count"]
    candidate_features["creator_flag"] = user_row["creator_flag"]

    candidate_features["user_prior_interactions"] = (
        user_prior_interactions
    )

    candidate_features["user_prior_clicks"] = (
        user_prior_clicks
    )

    candidate_features["user_prior_engagements"] = (
        user_prior_engagements
    )

    candidate_features["user_prior_watch_time"] = (
        user_prior_watch_time
    )

    candidate_features["user_prior_completions"] = (
        user_prior_completions
    )

    candidate_features["user_prior_ctr"] = (
        user_prior_ctr
    )

    candidate_features["user_prior_engagement_rate"] = (
        user_prior_engagement_rate
    )

    candidate_features["user_prior_avg_watch_time"] = (
        user_prior_avg_watch_time
    )

    candidate_features["user_prior_avg_completion"] = (
        user_prior_avg_completion
    )

    candidate_features["user_prior_active_days"] = (
        user_prior_active_days
    )

    candidate_features["user_prior_unique_content"] = (
        user_prior_unique_content
    )

    candidate_features["user_prior_unique_creators"] = (
        user_prior_unique_creators
    )

    # --------------------------------------------------------
    # 19. CREATOR TYPE
    # --------------------------------------------------------

    creator_type_map = (
        creators.set_index("creator_id")["creator_type"]
        .to_dict()
    )

    candidate_features["creator_type"] = (
        candidate_features["creator_id"]
        .map(creator_type_map)
        .fillna("Unknown")
    )

    # --------------------------------------------------------
    # 20. CREATOR FOLLOWERS
    # --------------------------------------------------------

    creator_followers_map = (
        creators.set_index("creator_id")["followers"]
        .to_dict()
    )

    candidate_features["creator_followers"] = (
        candidate_features["creator_id"]
        .map(creator_followers_map)
    )

    candidate_features["creator_followers"] = (
        candidate_features["creator_followers"]
        .fillna(0)
    )

    # --------------------------------------------------------
    # 21. FINAL 53-FEATURE CONTRACT
    # --------------------------------------------------------

    missing_features = [
        col
        for col in feature_columns
        if col not in candidate_features.columns
    ]

    if missing_features:
        raise ValueError(
            f"Missing model features: {missing_features}"
        )

    model_features = candidate_features[
        feature_columns
    ].copy()

    # --------------------------------------------------------
    # 22. FINAL MISSING VALUE SAFETY CHECK
    # --------------------------------------------------------

    numeric_columns = model_features.select_dtypes(
        include=[np.number]
    ).columns

    model_features[numeric_columns] = (
        model_features[numeric_columns]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    categorical_columns = model_features.select_dtypes(
        include=["object", "category"]
    ).columns

    for col in categorical_columns:
        model_features[col] = (
            model_features[col]
            .fillna("Unknown")
        )

    # --------------------------------------------------------
    # 23. MODEL SCORING
    # --------------------------------------------------------

    candidate_features["engagement_score"] = (
        random_forest.predict_proba(model_features)[:, 1]
    )

    # --------------------------------------------------------
    # 24. FINAL RANKING
    # --------------------------------------------------------

    candidate_features["engagement_norm"] = (
        candidate_features["engagement_score"]
        - candidate_features["engagement_score"].min()
    )

    engagement_range = (
        candidate_features["engagement_score"].max()
        - candidate_features["engagement_score"].min()
    )

    if engagement_range > 0:
        candidate_features["engagement_norm"] /= engagement_range
    else:
        candidate_features["engagement_norm"] = 0.5

    candidate_features["freshness_norm"] = (
        candidate_features["freshness"]
        .clip(0, 1)
    )

    # Small deterministic exploration component
    candidate_features["exploration_score"] = (
        rng.random(len(candidate_features))
    )

    candidate_features["final_score"] = (
        0.75 * candidate_features["engagement_norm"]
        + 0.15 * candidate_features["freshness_norm"]
        + 0.10 * candidate_features["exploration_score"]
    )

    # --------------------------------------------------------
    # 25. GREEDY DIVERSITY SELECTION
    # --------------------------------------------------------

    ranked = candidate_features.sort_values(
        "final_score",
        ascending=False
    ).copy()

    recommendations = []
    used_creators = set()
    used_genres = set()

    # First pass: maximize creator/genre diversity
    for _, row in ranked.iterrows():

        content_id = str(row["content_id"])
        creator_id = str(row["creator_id"])
        genre = str(row["genre"])

        if content_id in recommendations:
            continue

        creator_penalty = (
            0.10
            if creator_id in used_creators
            else 0.0
        )

        genre_penalty = (
            0.05
            if genre in used_genres
            else 0.0
        )

        adjusted_score = (
            row["final_score"]
            - creator_penalty
            - genre_penalty
        )

        recommendations.append(
            (adjusted_score, content_id, creator_id, genre)
        )

        if len(recommendations) >= k:
            break

        used_creators.add(creator_id)
        used_genres.add(genre)

    # Sort selected recommendations again by adjusted score
    recommendations = sorted(
        recommendations,
        key=lambda x: x[0],
        reverse=True
    )

    result = [
        item[1]
        for item in recommendations[:k]
    ]

    # --------------------------------------------------------
    # 26. FINAL OUTPUT CONTRACT
    # --------------------------------------------------------

    result = list(
        dict.fromkeys(
            str(content_id)
            for content_id in result
        )
    )

    return result[:k]


print("✓ Final cold-start-aware recommender defined")
print("✓ New-user fallback supported")
print("✓ New-content handling supported")
print("✓ New-creator handling supported")
print("✓ Existing-user ML ranking supported")
print("✓ No interaction_id dependency")
print("✓ Output contract: list[str]")

✓ Final cold-start-aware recommender defined
✓ New-user fallback supported
✓ New-content handling supported
✓ New-creator handling supported
✓ Existing-user ML ranking supported
✓ No interaction_id dependency
✓ Output contract: list[str]


In [281]:
# ============================================================
# FINAL RECOMMENDER SIGN-OFF TEST
# ============================================================

print("=" * 60)
print("FINAL RECOMMENDER SIGN-OFF")
print("=" * 60)

# ------------------------------------------------------------
# TEST 1: NEW USER
# ------------------------------------------------------------

new_users = set(users["user_id"]) - set(
    historical_interactions["user_id"]
)

new_user = sorted(new_users)[0]

new_user_recs = recommend_ml(
    new_user,
    k=10
)

print("\n[1] NEW USER")
print("User:", new_user)
print("Recommendations:", new_user_recs)

assert isinstance(new_user_recs, list)
assert len(new_user_recs) == 10
assert all(isinstance(x, str) for x in new_user_recs)
assert len(set(new_user_recs)) == 10

print("✓ New-user cold start passed")


# ------------------------------------------------------------
# TEST 2: EXISTING USER
# ------------------------------------------------------------

existing_user = (
    historical_interactions["user_id"]
    .value_counts()
    .index[0]
)

existing_user_recs = recommend_ml(
    existing_user,
    k=10
)

existing_seen = set(
    historical_interactions.loc[
        historical_interactions["user_id"] == existing_user,
        "content_id"
    ]
)

print("\n[2] EXISTING USER")
print("User:", existing_user)
print("Recommendations:", existing_user_recs)

assert isinstance(existing_user_recs, list)
assert len(existing_user_recs) == 10
assert all(isinstance(x, str) for x in existing_user_recs)
assert len(set(existing_user_recs)) == 10
assert not set(existing_user_recs).intersection(existing_seen)

print("✓ Existing-user personalization passed")
print("✓ No previously seen content recommended")


# ------------------------------------------------------------
# TEST 3: NEW CONTENT
# ------------------------------------------------------------

historical_content = set(
    historical_interactions["content_id"]
)

eligible_content_ids = set(
    eligible_content["content_id"]
)

new_content_set = (
    eligible_content_ids
    - historical_content
)

print("\n[3] NEW CONTENT")
print(
    "Eligible content with zero historical interactions:",
    len(new_content_set)
)

assert len(new_content_set) > 0

print("✓ New-content candidates exist")
print("✓ Cold-start feature defaults are available")


# ------------------------------------------------------------
# TEST 4: FEATURE CONTRACT
# ------------------------------------------------------------

assert len(feature_columns) == 53

print("\n[4] MODEL FEATURE CONTRACT")
print("Expected features:", len(feature_columns))
print("✓ 53-feature model contract preserved")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("✓ NOTEBOOK 07 RECOMMENDER SIGN-OFF PASSED")
print("=" * 60)

FINAL RECOMMENDER SIGN-OFF

[1] NEW USER
User: U00005
Recommendations: ['CT009210', 'CT002215', 'CT001900', 'CT002295', 'CT002868', 'CT008155', 'CT007293', 'CT005537', 'CT001930', 'CT006963']
✓ New-user cold start passed

[2] EXISTING USER
User: U04142
Recommendations: ['CT000003', 'CT008066', 'CT007942', 'CT008393', 'CT006977', 'CT003941', 'CT001712', 'CT004834', 'CT000299', 'CT004108']
✓ Existing-user personalization passed
✓ No previously seen content recommended

[3] NEW CONTENT
Eligible content with zero historical interactions: 21
✓ New-content candidates exist
✓ Cold-start feature defaults are available

[4] MODEL FEATURE CONTRACT
Expected features: 53
✓ 53-feature model contract preserved

✓ NOTEBOOK 07 RECOMMENDER SIGN-OFF PASSED
